# snake-arena · ACER

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/voaneves/snake-arena/blob/main/notebooks/05_acer.ipynb)

Retrace(λ), IS truncado com correção de viés, região de confiança.

**Este notebook é autocontido.** Não precisa clonar nada: o ambiente, a rede, o protocolo
de avaliação e o agente estão todos aqui dentro. O código do núcleo é **gerado a partir do
pacote** ([`voaneves/snake-arena`](https://github.com/voaneves/snake-arena)) e é byte a byte igual
em todos os notebooks — é isso que torna as curvas comparáveis.

`Runtime → Change runtime type → GPU (T4)` antes de rodar.

Assinatura do código gerado: `468b03afedee1eeb`


In [ ]:
# @title Ambiente
import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import json, math, time, glob, csv, platform, subprocess, sys, shutil, argparse
from dataclasses import dataclass, field, asdict

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers, ops, regularizers

print("TensorFlow", tf.__version__, "| Keras", keras.__version__,
      "| backend", keras.backend.backend())
GPUS = tf.config.list_physical_devices("GPU")
print("GPU:", GPUS or "nenhuma — vai rodar em CPU, muito mais lento")
for g in GPUS:
    tf.config.experimental.set_memory_growth(g, True)


## O núcleo, gerado a partir do pacote

A célula abaixo é **gerada**. Editá-la aqui não muda o repositório e faz o teste
`tests/test_notebooks.py` acusar divergência — o que é de propósito: é o que garante que
os 10 notebooks rodem exatamente o mesmo jogo, com a mesma régua.

Para mudar algo aqui, mude no pacote e rode `python tools/gerar_notebooks.py`.


In [ ]:
# ==== GERADO A PARTIR DO PACOTE — NÃO EDITE AQUI ====
# assinatura: 468b03afedee1eeb

from __future__ import annotations

# --- snakeai/env/vec_snake.py ---
"""`VecSnake` — Snake vetorizado, N tabuleiros independentes evoluindo em lote.

Este módulo é **a fonte única de verdade do ambiente**. Todo algoritmo do `snake-arena`
treina e é avaliado aqui, sem exceção — é isso que torna as curvas comparáveis. Ele não
importa TensorFlow nem Keras: é NumPy puro, roda em qualquer lugar e é rápido o bastante
para que o gargalo do treino seja a GPU, não o jogo.

O truque que faz ser rápido: em vez de uma lista de posições por cobra, guardamos uma
grade `occ` de inteiros onde `occ[n, y, x]` é **quantos passos faltam para aquela célula
ficar livre**. A cabeça recebe `occ = comprimento`; a cada passo o mundo inteiro decrementa
em 1 e a cauda some sozinha. Tudo vira operação NumPy em lote sobre `(N, B, B)` — nada de
laço Python por cobra.

Como bônus, essa grade *já é* a feature mais informativa que existe para Snake: normalizada
por comprimento, ela diz à rede **quando** cada célula vai desocupar, que é exatamente a
informação necessária para a cobra passar rente ao próprio corpo sem se prender.

Convenções fixadas pelo contrato de comparabilidade (`docs/COMPARABILITY.md`):

* tabuleiro 10x10, `starve_base = 100`;
* observação `(N, B, B, 5)` egocêntrica;
* 3 ações relativas com máscara de morte imediata;
* recompensa `+1` comer, `-1` morrer, `0` passo;
* **score = comida comida**, começando em zero. Nunca comprimento.
"""


import numpy as np

__all__ = ["VecSnake", "DIRS", "TURN", "N_ACTIONS", "N_CHANNELS", "DEFAULT_SEED"]

# Direções: 0=cima(-y), 1=direita(+x), 2=baixo(+y), 3=esquerda(-x)  (sentido horário)
DIRS = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]], dtype=np.int32)
# Ações relativas: 0=vira à esquerda, 1=segue reto, 2=vira à direita
TURN = np.array([-1, 0, 1], dtype=np.int32)

N_ACTIONS = 3
N_CHANNELS = 5
DEFAULT_SEED = 42


class VecSnake:
    """`num_envs` tabuleiros independentes de Snake evoluindo em lote.

    Observação: `(num_envs, B, B, 5)` float32, **egocêntrica** — o tabuleiro é rotacionado
    para que a cobra sempre olhe para cima. Isso colapsa as 4 simetrias de rotação e deixa
    a rede ~4x mais eficiente em amostras.

    Canais
    ------
    0. corpo (binário, sem a cabeça)
    1. cabeça
    2. decaimento da cauda: `occ / comprimento` em (0, 1]
    3. comida
    4. plano constante = comprimento / B**2  (a rede precisa saber o quão longa está)

    Parâmetros
    ----------
    num_envs : int
        Quantos tabuleiros correm em paralelo.
    board_size : int
        Lado do tabuleiro. O contrato oficial usa 10.
    starve_base : int, opcional
        Paciência base antes de morrer de fome; o limite efetivo é
        `starve_base + 2 * comprimento`. Padrão: `board_size ** 2`.
    rng : np.random.Generator, opcional
        Gerador próprio. Passe um com semente fixa para reprodutibilidade.
    """

    def __init__(self, num_envs=256, board_size=10, starve_base=None, rng=None):
        if board_size < 6:
            raise ValueError("tabuleiro pequeno demais para o corpo inicial (mínimo 6)")
        self.n = int(num_envs)
        self.b = int(board_size)
        self.cells = self.b * self.b
        self.starve_base = self.cells if starve_base is None else int(starve_base)
        self.rng = rng if rng is not None else np.random.default_rng(DEFAULT_SEED)

        self.occ = np.zeros((self.n, self.b, self.b), dtype=np.int32)
        self.head = np.zeros((self.n, 2), dtype=np.int32)
        self.food = np.zeros((self.n, 2), dtype=np.int32)
        self.dir = np.zeros(self.n, dtype=np.int32)
        self.length = np.zeros(self.n, dtype=np.int32)
        self.steps = np.zeros(self.n, dtype=np.int32)
        self.hunger = np.zeros(self.n, dtype=np.int32)
        self.score = np.zeros(self.n, dtype=np.int32)

        self._reset_idx(np.arange(self.n))

    # ------------------------------------------------------------------- reset
    def _reset_idx(self, idx):
        """Reinicia apenas os ambientes em `idx`, em lote."""
        if idx.size == 0:
            return
        k = idx.size
        b = self.b
        self.occ[idx] = 0
        # cabeça longe das bordas para caber o corpo inicial de 3
        self.head[idx] = self.rng.integers(2, b - 2, size=(k, 2), dtype=np.int32)
        self.dir[idx] = self.rng.integers(0, 4, size=k, dtype=np.int32)
        self.length[idx] = 3
        self.steps[idx] = 0
        self.hunger[idx] = 0
        self.score[idx] = 0

        d = DIRS[self.dir[idx]]                       # (k, 2)
        for back, ttl in ((0, 3), (1, 2), (2, 1)):    # cabeça, meio, cauda
            p = self.head[idx] - back * d
            np.clip(p, 0, b - 1, out=p)
            self.occ[idx, p[:, 0], p[:, 1]] = ttl

        self._spawn_food(idx)

    def _spawn_food(self, idx):
        """Sorteia comida uniformemente entre as células livres (vetorizado)."""
        if idx.size == 0:
            return
        free = self.occ[idx].reshape(idx.size, -1) == 0
        r = self.rng.random((idx.size, self.cells))
        r[~free] = -1.0
        flat = r.argmax(axis=1)
        self.food[idx, 0] = flat // self.b
        self.food[idx, 1] = flat % self.b

    def reset(self):
        """Reinicia todos os ambientes. Retorna `(obs, mask)`."""
        self._reset_idx(np.arange(self.n))
        return self.obs(), self.action_mask()

    # -------------------------------------------------------------- observação
    def _raw_planes(self):
        """Os 5 canais no referencial do tabuleiro, antes da rotação egocêntrica."""
        b, n = self.b, self.n
        occ = self.occ
        body = (occ > 0).astype(np.float32)
        head = np.zeros((n, b, b), dtype=np.float32)
        rows = np.arange(n)
        head[rows, self.head[:, 0], self.head[:, 1]] = 1.0
        body -= head                                   # cabeça sai do canal de corpo
        decay = occ.astype(np.float32) / self.length[:, None, None].astype(np.float32)
        food = np.zeros((n, b, b), dtype=np.float32)
        food[rows, self.food[:, 0], self.food[:, 1]] = 1.0
        lenpl = np.broadcast_to(
            (self.length.astype(np.float32) / self.cells)[:, None, None], (n, b, b)
        )
        return np.stack([body, head, decay, food, lenpl], axis=-1)

    def obs(self):
        """Planos rotacionados para o referencial da cabeça (sempre olhando p/ cima)."""
        raw = self._raw_planes()
        out = np.empty_like(raw)
        for k in range(4):
            m = self.dir == k
            if m.any():
                out[m] = np.rot90(raw[m], k=k, axes=(1, 2))
        return out

    # ----------------------------------------------------------------- máscara
    def _next_head(self, actions):
        """Posição e direção da cabeça se `actions` fosse aplicada agora."""
        nd = (self.dir + TURN[actions]) % 4
        return self.head + DIRS[nd], nd

    def _lethal(self, pos):
        """True onde a posição mata (parede ou corpo que ainda não desocupou)."""
        b = self.b
        oob = (pos[:, 0] < 0) | (pos[:, 0] >= b) | (pos[:, 1] < 0) | (pos[:, 1] >= b)
        safe_pos = np.where(oob[:, None], 0, pos)
        # a cauda vai embora neste passo -> célula com occ<=1 estará livre
        hit = self.occ[np.arange(self.n), safe_pos[:, 0], safe_pos[:, 1]] > 1
        return oob | (hit & ~oob)

    def _raw_mask(self):
        """`(N, 3)` bool sem o *override* de beco sem saída — a verdade nua."""
        mask = np.empty((self.n, N_ACTIONS), dtype=bool)
        for a in range(N_ACTIONS):
            pos, _ = self._next_head(np.full(self.n, a, dtype=np.int32))
            mask[:, a] = ~self._lethal(pos)
        return mask

    def dead_ends(self):
        """`(N,)` bool: True onde **todas** as três ações matam.

        Existe porque `action_mask()` não permite descobrir isso — lá, um beco sem saída
        aparece como "tudo liberado". Quem precisa distinguir (testes, diagnóstico, o
        filtro de segurança) pergunta aqui.
        """
        return ~self._raw_mask().any(axis=1)

    def action_mask(self):
        """`(N, 3)` bool: True = ação não mata imediatamente.

        Se as três matam, liberamos todas (a cobra morreu de qualquer jeito) — assim a
        distribuição nunca fica sem suporte e o log-prob não vira NaN. Use `dead_ends()`
        para saber quando esse caso ocorreu.
        """
        mask = self._raw_mask()
        mask[~mask.any(axis=1)] = True
        return mask

    # -------------------------------------------------------------------- step
    def step(self, actions, shaping_coef=0.0, gamma=0.99):
        """Avança todos os ambientes um passo.

        Retorna `(obs, mask, reward, done, info)`. Ambientes terminados são resetados
        automaticamente; `obs` já é o do episódio novo, e `info` guarda as estatísticas
        do episódio que acabou.

        `info` contém:
            scores      : score final dos episódios encerrados neste passo
            lengths     : duração em passos desses episódios
            wins        : quantos encheram o tabuleiro
            deaths      : quantos morreram por colisão
            starved     : quantos foram truncados por fome
            trunc_idx   : índices dos truncados por fome
            final_obs   : observação terminal dos truncados (para bootstrap do valor)
            final_mask  : máscara terminal dos truncados
        """
        n, b = self.n, self.b
        rows = np.arange(n)
        actions = np.asarray(actions, dtype=np.int32)

        d_old = np.abs(self.head - self.food).sum(axis=1).astype(np.float32)

        new_head, new_dir = self._next_head(actions)
        dead = self._lethal(new_head)
        new_head = np.where(dead[:, None], self.head, new_head)  # congela quem morreu

        ate = (
            (~dead)
            & (new_head[:, 0] == self.food[:, 0])
            & (new_head[:, 1] == self.food[:, 1])
        )

        # cauda anda quando não comeu
        moved = ~ate & ~dead
        self.occ[moved] = np.maximum(self.occ[moved] - 1, 0)

        self.length += ate.astype(np.int32)
        self.score += ate.astype(np.int32)
        alive = ~dead
        self.head[alive] = new_head[alive]
        self.dir[alive] = new_dir[alive]
        self.occ[rows[alive], self.head[alive, 0], self.head[alive, 1]] = self.length[alive]

        self.steps += 1
        self.hunger = np.where(ate, 0, self.hunger + 1)

        won = self.length >= self.cells
        need_food = ate & ~won
        if need_food.any():
            self._spawn_food(np.nonzero(need_food)[0])

        starve_limit = self.starve_base + 2 * self.length
        starved = (self.hunger >= starve_limit) & ~dead & ~won

        # ---- recompensa
        reward = np.zeros(n, dtype=np.float32)
        reward += ate.astype(np.float32)
        reward -= dead.astype(np.float32)
        reward += won.astype(np.float32) * 2.0
        reward -= starved.astype(np.float32) * 0.5
        if shaping_coef > 0.0:
            # o delta só faz sentido quando a comida não mudou de lugar
            d_new = np.abs(self.head - self.food).sum(axis=1).astype(np.float32)
            phi_old = -d_old / b
            phi_new = -d_new / b
            delta = np.where(dead | won | ate, 0.0, gamma * phi_new - phi_old)
            reward += shaping_coef * delta

        done = dead | won | starved

        # Truncamento por fome: o episódio *continuaria*, então precisamos do valor do
        # estado final para fazer bootstrap. Como o env reseta sozinho, guardamos a
        # observação terminal antes do reset (custa uma passada extra, só quando ocorre).
        starved_idx = np.nonzero(starved)[0]
        final_obs = final_mask = None
        if starved_idx.size:
            final_obs = self.obs()[starved_idx]
            final_mask = self.action_mask()[starved_idx]

        info = {
            "scores": self.score[done].copy(),
            "lengths": self.steps[done].copy(),
            "wins": int(won.sum()),
            "deaths": int(dead.sum()),
            "starved": int(starved.sum()),
            "trunc_idx": starved_idx,
            "final_obs": final_obs,
            "final_mask": final_mask,
        }
        self._reset_idx(np.nonzero(done)[0])
        return self.obs(), self.action_mask(), reward, done, info

    # -------------------------------------------------------------- utilidades
    def free_space_from(self, env_i, pos):
        """Flood-fill: quantas células livres são alcançáveis a partir de `pos`.

        Usado só no filtro de segurança da inferência, nunca no treino.
        """
        b = self.b
        occ = self.occ[env_i]
        seen = np.zeros((b, b), dtype=bool)
        stack = [(int(pos[0]), int(pos[1]))]
        seen[pos[0], pos[1]] = True
        count = 0
        while stack:
            y, x = stack.pop()
            count += 1
            for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                ny, nx = y + dy, x + dx
                if 0 <= ny < b and 0 <= nx < b and not seen[ny, nx] and occ[ny, nx] <= 1:
                    seen[ny, nx] = True
                    stack.append((ny, nx))
        return count

    # -------------------------------------------------------- estado serializável
    #: Os campos que definem completamente o estado do jogo. Nada fora desta lista
    #: influencia o futuro — é o que torna a busca em árvore possível.
    CAMPOS_ESTADO = ("occ", "head", "food", "dir", "length", "steps", "hunger", "score")

    def get_state(self):
        """Cópia do estado de todos os ambientes, como dicionário de arrays.

        Existe para a busca em árvore: o MCTS precisa voltar a um nó anterior, e a única
        forma honesta de fazer isso é restaurar o estado exato. Snake é determinístico e de
        informação perfeita, então este dicionário *é* o nó da árvore.
        """
        return {c: getattr(self, c).copy() for c in self.CAMPOS_ESTADO}

    def set_state(self, estado):
        """Restaura o estado. Não valida por desempenho — use `check_invariants` em teste."""
        for c in self.CAMPOS_ESTADO:
            getattr(self, c)[...] = estado[c]
        return self

    def estado_de(self, i):
        """O estado de um único ambiente, como dicionário de arrays sem eixo de lote."""
        return {c: getattr(self, c)[i].copy() for c in self.CAMPOS_ESTADO}

    def escrever_estado(self, i, estado_unico):
        for c in self.CAMPOS_ESTADO:
            getattr(self, c)[i] = estado_unico[c]

    # ------------------------------------------------------------- introspecção
    def check_invariants(self):
        """Levanta `AssertionError` se o estado interno estiver inconsistente.

        Barato o bastante para rodar em testes e em depuração; nunca no laço de treino.
        """
        assert (self.occ >= 0).all(), "occ negativo"
        assert ((self.occ > 0).sum(axis=(1, 2)) == self.length).all(), \
            "número de células ocupadas não bate com o comprimento"
        assert (self.occ.reshape(self.n, -1).max(axis=1) == self.length).all(), \
            "a cabeça deveria ser a célula de maior occ"
        rows = np.arange(self.n)
        assert (self.occ[rows, self.head[:, 0], self.head[:, 1]] == self.length).all(), \
            "occ na posição da cabeça não é o comprimento"
        occupied_food = self.occ[rows, self.food[:, 0], self.food[:, 1]] > 0
        assert not occupied_food.any() or (self.length >= self.cells).any(), \
            "comida dentro do corpo"
        assert (self.score == self.length - 3).all(), \
            "score deve ser comprimento - 3"

    def __repr__(self):
        return (
            f"VecSnake(num_envs={self.n}, board_size={self.b}, "
            f"starve_base={self.starve_base})"
        )


# --- snakeai/otimizadores.py ---
"""Otimizadores — o eixo de ablação de primeira ordem.

Onde foi parar o K-FAC
----------------------
Quatro notebooks do `colab-rl` tentaram K-FAC e nenhum roda: dependiam de
`tensorflow.contrib.kfac`, que não existe desde o TensorFlow 2. Ele **voltou**, mas não
para cá: mora em `snakeai/kfac.py` e é usado pelo `ACKTR` (`snakeai/agents/acktr.py`).

O motivo de não estar neste eixo é estrutural, não histórico. `cria_otimizador` recebe um
nome e um learning rate; um `keras.optimizers.Optimizer` recebe pares `(gradiente,
variável)`. O K-FAC precisa das **ativações de entrada** e dos **gradientes de
pré-ativação** de cada camada — coisas que só existem durante o forward/backward e que
nenhum otimizador do Keras enxerga. Espremê-lo nesta assinatura exigiria refazer o forward
por dentro do otimizador, que foi o que a API Keras do `tensorflow/kfac` fazia (arquivada
em 19/04/2026).

A **pergunta** por trás daqueles notebooks continua sendo boa: *o otimizador importa?* Este
módulo é a resposta de primeira ordem — um eixo de ablação com otimizadores que existem,
funcionam em Keras 3 e cobrem escolhas de projeto diferentes. A resposta de segunda ordem é
a curva do ACKTR ao lado da do A2C, que é o mesmo algoritmo com a curvatura ligada:

===========  ==============================================================
nome         o que muda
===========  ==============================================================
``rmsprop``  o que o repositório antigo usava na maioria dos experimentos
``adam``     momento + escala adaptativa; o padrão de fato em RL
``adamw``    Adam com decaimento de peso desacoplado — regulariza sem mexer
             na escala adaptativa, ao contrário do `weight_decay` clássico
``lion``     só o **sinal** do momento; usa muito menos memória de estado e
             costuma preferir LR ~10x menor
``sgd``      o controle: momento e nada mais. Se o eixo não separar nada,
             este aqui denuncia
===========  ==============================================================

Todos entram pelo mesmo lugar: `cfg.optimizer = "adamw"`. O resto do experimento não muda,
que é o que torna a comparação uma ablação e não uma anedota.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras

__all__ = ["OTIMIZADORES", "cria_otimizador", "LR_SUGERIDO"]

OTIMIZADORES = ("adam", "adamw", "rmsprop", "lion", "sgd")

#: Multiplicador de learning rate típico de cada otimizador, relativo ao Adam. O Lion usa
#: só o sinal do momento, então o passo tem magnitude constante e o LR precisa ser bem
#: menor; o SGD, sem escala adaptativa, precisa de bem maior. Comparar otimizadores com o
#: mesmo LR não mede otimizador — mede quem tolera aquele LR específico.
LR_SUGERIDO = {"adam": 1.0, "adamw": 1.0, "rmsprop": 1.0, "lion": 0.1, "sgd": 30.0}


def cria_otimizador(nome, learning_rate, clipnorm=None, weight_decay=1e-4, **kw):
    """Devolve um `keras.optimizers.Optimizer` pelo nome.

    `learning_rate` é o valor **base**; aplique `LR_SUGERIDO[nome]` por fora se quiser a
    escala típica de cada um. Deixar isso explícito é de propósito: um experimento que
    ajusta o LR junto com o otimizador está medindo os dois ao mesmo tempo, e precisa
    dizer isso.
    """
    nome = nome.lower()
    comum = {"learning_rate": learning_rate}
    if clipnorm is not None:
        comum["clipnorm"] = clipnorm

    if nome == "adam":
        return keras.optimizers.Adam(epsilon=1e-5, **comum, **kw)
    if nome == "adamw":
        return keras.optimizers.AdamW(epsilon=1e-5, weight_decay=weight_decay,
                                      **comum, **kw)
    if nome == "rmsprop":
        return keras.optimizers.RMSprop(rho=0.95, epsilon=1e-5, **comum, **kw)
    if nome == "lion":
        return keras.optimizers.Lion(beta_1=0.9, beta_2=0.99, **comum, **kw)
    if nome == "sgd":
        return keras.optimizers.SGD(momentum=0.9, nesterov=True, **comum, **kw)
    raise ValueError(f"otimizador desconhecido: {nome!r}. Use um de {OTIMIZADORES}")


# --- snakeai/eval.py ---
"""Avaliação — o protocolo oficial do benchmark.

Este módulo responde à única pergunta que importa: **quanto esse agente tira, de verdade?**
Ele é deliberadamente independente de TensorFlow e Keras — recebe uma *função de política*,
não um modelo. Isso permite avaliar qualquer coisa pelo mesmo caminho: uma rede Keras, uma
tabela, uma heurística escrita à mão, ou a política aleatória que define o piso. E permite
testar a avaliação sem GPU.

Protocolo fixado pelo contrato de comparabilidade (`docs/COMPARABILITY.md`):

* 1.000 episódios, tabuleiro 10x10;
* política **greedy** (sem exploração);
* `seed = 123`;
* **sem** filtro de segurança na curva principal;
* métrica = `score` (comida comida), nunca comprimento.

Sobre o viés que este módulo corrige
------------------------------------
A forma ingênua de avaliar é rodar N ambientes em paralelo e parar assim que 1.000
episódios terminarem. Isso **subestima o agente**: episódios curtos terminam primeiro e
entram na amostra, enquanto os longos — que são justamente os bons — ainda estão correndo
quando a contagem fecha. Quanto melhor o agente, pior o viés.

A correção é simples: cada ambiente contribui com o mesmo número de episódios (os
primeiros que ele terminar), em vez de a amostra ser "os primeiros a terminar no total".
"""


import math

import numpy as np


__all__ = [
    "MASK_NEG",
    "evaluate",
    "random_baseline",
    "random_policy",
    "keras_policy",
    "apply_safety_filter",
    "verdict",
]

MASK_NEG = -1e9

#: Piso documentado no README: política aleatória com máscara, 1.000 episódios, 10x10.
PISO_ALEATORIO_10X10 = 1.08


# --------------------------------------------------------------------- políticas
def random_policy(rng=None):
    """Política uniforme sobre as ações permitidas — o piso do benchmark.

    Não é "aleatória pura": ela respeita a máscara, ou seja, já evita a morte imediata.
    É o piso honesto, porque qualquer agente do benchmark também tem a máscara.
    """
    rng = rng if rng is not None else np.random.default_rng(0)

    def politica(obs, mask):
        return np.where(mask, rng.random(mask.shape), -np.inf).astype(np.float32)

    return politica


def keras_policy(model, batch_size=None):
    """Embrulha um modelo Keras (actor-critic) numa função de política.

    O import de TensorFlow acontece aqui dentro, de propósito: quem só quer avaliar uma
    heurística não precisa ter TF instalado.
    """
    import tensorflow as tf  # noqa: PLC0415  (lazy de propósito)

    @tf.function(reduce_retracing=True)
    def _forward(obs, mask):
        saida = model(obs, training=False)
        logits = saida[0] if isinstance(saida, (list, tuple)) else saida
        return tf.where(mask, logits, tf.fill(tf.shape(logits), MASK_NEG))

    def politica(obs, mask):
        return _forward(
            tf.convert_to_tensor(obs), tf.convert_to_tensor(mask)
        ).numpy()

    return politica


# ------------------------------------------------------------- filtro de segurança
def apply_safety_filter(env: VecSnake, logits, margin=1.0, penalty=50.0):
    """Penaliza ações que deixariam a cobra num bolso menor que o próprio corpo.

    Pós-processamento de inferência, **não aprendido**: entre as ações que a rede
    considera boas, desencoraja as que se fecham num espaço sem saída (flood-fill a partir
    da nova cabeça). Por isso ele nunca entra na curva principal do benchmark — vira
    coluna separada da tabela.

    A penalidade é grande mas finita: se *todas* as opções forem ruins, a ordem relativa
    que a rede preferia é preservada e o agente escolhe a menos pior.
    """
    out = np.array(logits, dtype=np.float32, copy=True)
    for a in range(N_ACTIONS):
        pos, _ = env._next_head(np.full(env.n, a, dtype=np.int32))
        lethal = env._lethal(pos)
        for i in range(env.n):
            if lethal[i]:
                out[i, a] = MASK_NEG
            elif env.free_space_from(i, pos[i]) < margin * env.length[i]:
                out[i, a] -= penalty
    return out


# ------------------------------------------------------------------- avaliação
def evaluate(
    policy,
    board_size=10,
    episodes=1000,
    num_envs=250,
    greedy=True,
    safety=False,
    seed=123,
    max_steps=200_000,
    rng=None,
):
    """Roda o protocolo oficial e devolve `(stats, scores)`.

    Parâmetros
    ----------
    policy : callable
        `policy(obs, mask) -> logits (N, 3)`. Já deve aplicar a máscara aos logits;
        `evaluate` não confia nisso e reaplica de qualquer forma.
    episodes : int
        Quantos episódios compõem a amostra. O contrato usa 1.000.
    greedy : bool
        `True` = argmax (o padrão do benchmark). `False` = amostra da softmax.
    safety : bool
        Liga o flood-fill. Fora da curva principal, por construção.
    seed : int
        Semente do ambiente. Fixa em 123 no contrato, para que a sequência de comidas
        seja a mesma para todos os algoritmos.

    Cada ambiente contribui com o mesmo número de episódios — ver a nota sobre viés no
    topo do módulo.
    """
    env = VecSnake(num_envs, board_size, rng=np.random.default_rng(seed))
    rng = rng if rng is not None else np.random.default_rng(seed + 1)
    obs, mask = env.reset()
    apos_passo = getattr(policy, "apos_passo", None)

    por_env = math.ceil(episodes / num_envs)
    coletados = [[] for _ in range(num_envs)]
    #: Por que cada episódio da amostra terminou. Score sozinho não distingue "o agente
    #: joga mal" de "o agente anda em círculo": um DQN greedy no começo do treino tira
    #: 0,05 morrendo **100% por fome**, e a leitura correta disso não é "não aprendeu", é
    #: "a política determinística entrou em ciclo". São problemas diferentes.
    motivos = {"fome": 0, "colisao": 0, "tabuleiro_cheio": 0}
    faltam = num_envs
    passos = 0

    while faltam > 0 and passos < max_steps:
        logits = np.asarray(policy(obs, mask), dtype=np.float32)
        logits = np.where(mask, logits, MASK_NEG)
        if safety:
            logits = apply_safety_filter(env, logits)

        if greedy:
            acoes = logits.argmax(axis=1).astype(np.int32)
        else:
            z = logits - logits.max(axis=1, keepdims=True)
            p = np.exp(z)
            p /= p.sum(axis=1, keepdims=True)
            acoes = (p.cumsum(axis=1) > rng.random((num_envs, 1))).argmax(axis=1).astype(np.int32)

        obs, mask, r, done, info = env.step(acoes)
        passos += 1

        # Políticas com estado recorrente (DreamerV3) precisam saber o que de fato
        # aconteceu: a ação escolhida — que pode não ser o argmax, se o filtro de
        # segurança agiu — e onde o episódio terminou, para zerar o estado latente ali.
        # Políticas sem memória simplesmente não expõem este método.
        if apos_passo is not None:
            apos_passo(acoes, done)

        # `info["scores"]` é o score **final** do episódio, já contando a comida do
        # último passo. Ler `env.score` antes do passo perde exatamente um ponto nos
        # episódios que terminam comendo — que são precisamente as vitórias. Ver
        # `test_eval.py::test_a_winning_episode_scores_the_last_apple`.
        truncados = set(info["trunc_idx"].tolist())
        for j, i in enumerate(np.nonzero(done)[0]):
            if len(coletados[i]) < por_env:
                s_final = int(info["scores"][j])
                coletados[i].append(s_final)
                if i in truncados:
                    motivos["fome"] += 1
                elif s_final == board_size * board_size - 3:
                    motivos["tabuleiro_cheio"] += 1
                else:
                    motivos["colisao"] += 1
                if len(coletados[i]) == por_env:
                    faltam -= 1

    scores = np.array([s for lista in coletados for s in lista][:episodes], dtype=np.int32)
    if scores.size == 0:
        raise RuntimeError("nenhum episódio terminou — aumente `max_steps`")

    perfeito = board_size * board_size - 3
    # A taxa de vitória sai da **amostra coletada**, não de um contador do laço: o laço
    # continua rodando os ambientes que já cumpriram a cota, e somar as vitórias deles
    # daria uma taxa que não corresponde aos episódios de fato medidos.
    stats = {
        "episodes": int(scores.size),
        "score_mean": float(scores.mean()),
        "score_median": float(np.median(scores)),
        "score_std": float(scores.std()),
        "score_max": int(scores.max()),
        "score_p95": float(np.percentile(scores, 95)),
        "win_rate": float((scores == perfeito).mean()),
        "perfect_possible": perfeito,
        "env_steps_used": int(passos),
        "completo": bool(faltam == 0),
    }
    total_motivos = max(1, sum(motivos.values()))
    stats.update({f"fim_{k}": v / total_motivos for k, v in motivos.items()})
    return stats, scores


def random_baseline(board_size=10, episodes=1000, num_envs=250, seed=123):
    """O piso: política uniforme sobre as ações permitidas.

    É o número contra o qual todo resultado do benchmark é lido. Num 10x10 ele vale
    ~1,08 — qualquer coisa que não esteja bem acima disso não aprendeu nada.
    """
    stats, _ = evaluate(
        random_policy(np.random.default_rng(seed)),
        board_size=board_size,
        episodes=episodes,
        num_envs=num_envs,
        greedy=False,
        seed=seed,
    )
    return stats["score_mean"]


# ---------------------------------------------------------------------- veredito
def verdict(policy, board_size=10, episodes=1000, num_envs=250, com_filtro=True, seed=123):
    """A resposta objetiva para "aprendeu mesmo?".

    Roda, na mesma execução, três regimes e devolve a tabela:

    ===========================  =============================================
    regime                       o que mede
    ===========================  =============================================
    aleatório com máscara        o piso — quanto se tira sem aprender nada
    agente (greedy)              a política pura, sem nenhuma ajuda externa
    agente + filtro de segurança o teto prático, com o flood-fill ligado
    ===========================  =============================================

    Se a linha do meio não estiver bem acima do piso, não aprendeu — e aí o problema é de
    hiperparâmetro ou de tempo de treino, não do código.
    """
    linhas = []

    piso = random_baseline(board_size, episodes, num_envs, seed)
    linhas.append({"regime": "aleatório com máscara", "score_mean": piso})

    st, sc = evaluate(policy, board_size=board_size, episodes=episodes,
                      num_envs=num_envs, greedy=True, seed=seed)
    linhas.append({"regime": "agente (greedy)", "scores": sc, **st})

    if com_filtro:
        # o flood-fill é laço Python: menos ambientes, para não ficar lento
        stf, scf = evaluate(policy, board_size=board_size, episodes=episodes,
                            num_envs=min(num_envs, 64), greedy=True, safety=True,
                            seed=seed)
        linhas.append({"regime": "agente + filtro de segurança", "scores": scf, **stf})

    return {
        "piso": piso,
        "perfeito": board_size * board_size - 3,
        "ganho_sobre_o_piso": linhas[1]["score_mean"] / max(piso, 1e-9),
        "linhas": linhas,
    }


def format_verdict(resultado):
    """Formata o retorno de `verdict` como tabela de texto."""
    larg = 30
    out = [f"{'regime':<{larg}}{'média':>8}{'mediana':>9}{'máx':>6}{'cheio':>8}", "-" * (larg + 31)]
    for ln in resultado["linhas"]:
        med = f"{ln['score_median']:.0f}" if "score_median" in ln else "-"
        mx = f"{ln['score_max']}" if "score_max" in ln else "-"
        wr = f"{ln['win_rate']:.1%}" if "win_rate" in ln else "-"
        out.append(f"{ln['regime']:<{larg}}{ln['score_mean']:>8.2f}{med:>9}{mx:>6}{wr:>8}")
    out.append("-" * (larg + 31))
    out.append(
        f"score perfeito: {resultado['perfeito']}   |   "
        f"ganho sobre o piso: {resultado['ganho_sobre_o_piso']:.1f}x"
    )
    ag = resultado["linhas"][1]
    if "fim_fome" in ag:
        out.append(
            f"como terminou: fome {ag['fim_fome']:.0%} · colisão {ag['fim_colisao']:.0%}"
            f" · tabuleiro cheio {ag['fim_tabuleiro_cheio']:.0%}"
        )
        # Morrer de fome é o fim NORMAL aqui: a máscara de morte impede a colisão, então
        # até a política aleatória termina 85% dos episódios por fome. O que denuncia o
        # ciclo é a combinação — quase nenhuma colisão **e** score abaixo do piso, ou seja,
        # a cobra anda para sempre sem nunca comer.
        if ag["fim_colisao"] < 0.05 and ag["score_mean"] < resultado["piso"]:
            out.append(
                "  ⚠ nunca colide e não come: a política determinística entrou em ciclo.\n"
                "    Não é 'jogou mal' — é falta de exploração na hora de agir. Normal cedo\n"
                "    num DQN greedy, e é por isso que o score de TREINO (ε-greedy) fica\n"
                "    acima do de AVALIAÇÃO (greedy) nesta fase."
            )
    return "\n".join(out)


# --- snakeai/record.py ---
"""Registro de execuções — o esquema do `history.json` e o validador do contrato.

Este módulo é o porteiro do benchmark. Toda execução de todo algoritmo escreve o mesmo
arquivo, com os mesmos campos, e passa pela mesma validação antes de virar uma linha no
gráfico. **Um resultado que não valida não entra na arena** — não porque seja ruim, mas
porque não é comparável, que é pior.

A regra vale inclusive para as curvas históricas do `colab-rl`: elas são convertidas para
este mesmo esquema, mas com `comparable=False` e o motivo registrado em `caveat`. Assim
elas aparecem no gráfico como contexto (tracejado cinza) sem nunca serem confundidas com
um competidor.

Sem dependências além da biblioteca padrão e do NumPy — o validador roda no CI em segundos.
"""


import json
import os
import platform
import subprocess
import sys
import time
from dataclasses import asdict, dataclass, field

import numpy as np

__all__ = [
    "SCHEMA_VERSION",
    "CONTRATO",
    "ORCAMENTO_OFICIAL",
    "ContractViolation",
    "RunRecord",
    "Recorder",
    "validate",
    "save",
    "load",
    "load_all",
    "from_legacy_csv",
]

SCHEMA_VERSION = 1

#: Os valores que **todos** os resultados oficiais precisam compartilhar.
#: Espelha a tabela do README; mudar aqui é mudar o contrato, e invalida o histórico.
CONTRATO = {
    "env": "VecSnake",
    "board_size": 10,
    "starve_base": 100,
    "n_channels": 5,
    "n_actions": 3,
    "obs": "egocentric",
    "metric": "score",
    "reward_food": 1.0,
    "reward_death": -1.0,
    "eval_episodes": 1000,
    "eval_seed": 123,
    "eval_greedy": True,
    "eval_safety": False,
}

#: Orçamento oficial, em passos de ambiente. Fica fora do `CONTRATO` porque não descreve o
#: ambiente, mas é igualmente obrigatório: comparar um algoritmo que treinou 5 M passos com
#: outro que treinou 500 mil não mede algoritmo, mede paciência. Validado a partir de
#: `config["total_steps"]`.
ORCAMENTO_OFICIAL = 5_000_000

#: Piso e teto do 10x10, medidos e documentados no README.
PISO_ALEATORIO = 1.21
SCORE_PERFEITO = 97


class ContractViolation(Exception):
    """Levantada quando um registro não obedece ao contrato de comparabilidade."""


# ------------------------------------------------------------------- estrutura
@dataclass
class RunRecord:
    """Uma execução completa de um algoritmo, com curva e resultado final.

    Campos
    ------
    algo, variant, seed
        Identidade da execução. `runs/<algo>/<variant>/seed<N>/history.json`.
    net, params
        Tronco usado e número de parâmetros treináveis — o eixo "arquitetura importa?".
    config
        Hiperparâmetros do agente, como dicionário livre. Não é validado; é documentação.
    env_spec
        O recorte do contrato que esta execução usou. **É** validado.
    curve
        Lista de pontos ao longo do treino. Cada ponto tem, no mínimo, `global_step`;
        `eval_score_mean` aparece só nos passos em que a avaliação rodou.
    final
        O `stats` devolvido por `snakeai.eval.evaluate` no fim.
    comparable, caveat
        `False` marca uma curva que entra no gráfico como contexto histórico, com o
        motivo em `caveat`. Toda execução nova nasce `True`.
    """

    algo: str
    variant: str = "default"
    seed: int = 0
    net: str = ""
    params: int = 0
    config: dict = field(default_factory=dict)
    env_spec: dict = field(default_factory=lambda: dict(CONTRATO))
    curve: list = field(default_factory=list)
    final: dict = field(default_factory=dict)
    comparable: bool = True
    caveat: str = ""
    meta: dict = field(default_factory=dict)
    schema_version: int = SCHEMA_VERSION

    # ---------------------------------------------------------------- derivados
    @property
    def run_id(self):
        return f"{self.algo}/{self.variant}/seed{self.seed}"

    @property
    def rel_path(self):
        return os.path.join("runs", self.algo, self.variant, f"seed{self.seed}",
                            "history.json")

    def steps(self):
        return np.array([p["global_step"] for p in self.curve], dtype=np.int64)

    @property
    def oficial(self):
        """Pode competir na arena? Comparável **e** sem violação de contrato registrada.

        Separado de `comparable` de propósito: uma execução de fumaça não é uma curva
        histórica. Ela não compete, mas também não vira contexto — simplesmente não
        aparece, e o motivo fica em `meta["contract_violations"]`.
        """
        return self.comparable and not self.meta.get("contract_violations")

    def eval_curve(self):
        """`(passos, scores)` só dos pontos em que a avaliação rodou."""
        pts = [p for p in self.curve if p.get("eval_score_mean") is not None]
        x = np.array([p["global_step"] for p in pts], dtype=np.int64)
        y = np.array([p["eval_score_mean"] for p in pts], dtype=np.float64)
        return x, y


# -------------------------------------------------------------------- gravação
class Recorder:
    """Acumula a curva durante o treino e grava o `history.json` no fim.

    Uso típico, dentro do laço de treino::

        rec = Recorder("ppo", variant="resnet_small", seed=0, net="resnet_small",
                       params=model.count_params(), config=asdict(cfg))
        ...
        rec.log(global_step=n, episodes=e, train_score_mean=m)
        rec.log(global_step=n, eval_score_mean=stats["score_mean"])   # nos passos de eval
        ...
        rec.finish(stats)
        rec.save()            # valida antes de escrever; levanta se violar o contrato
    """

    def __init__(self, algo, variant="default", seed=0, net="", params=0,
                 config=None, env_spec=None, root="runs"):
        self.root = root
        self.t0 = time.perf_counter()
        self.record = RunRecord(
            algo=algo, variant=variant, seed=seed, net=net, params=int(params),
            config=dict(config or {}),
            env_spec=dict(env_spec or CONTRATO),
            meta=_ambiente(),
        )

    def log(self, global_step, **metrics):
        """Anexa um ponto à curva. `global_step` é o eixo oficial."""
        ponto = {"global_step": int(global_step),
                 "wall_s": round(time.perf_counter() - self.t0, 3)}
        for k, v in metrics.items():
            ponto[k] = _jsonable(v)
        self.record.curve.append(ponto)
        return ponto

    def finish(self, final_stats, comparable=True, caveat=""):
        self.record.final = {k: _jsonable(v) for k, v in dict(final_stats).items()}
        self.record.comparable = bool(comparable)
        self.record.caveat = str(caveat)
        self.record.meta["wall_s_total"] = round(time.perf_counter() - self.t0, 3)
        return self.record

    def save(self, path=None, skip_validation=False):
        if not skip_validation:
            problemas = validate(self.record)
            if problemas:
                raise ContractViolation(
                    f"{self.record.run_id} viola o contrato:\n  - "
                    + "\n  - ".join(problemas)
                )
        destino = path or os.path.join(self.root, self.record.algo,
                                       self.record.variant,
                                       f"seed{self.record.seed}", "history.json")
        return save(self.record, destino)


def save(record: RunRecord, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(asdict(record), f, ensure_ascii=False, indent=2)
    return path


def load(path) -> RunRecord:
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    d.pop("schema_version", None)
    return RunRecord(**d, schema_version=SCHEMA_VERSION)


def load_all(root="runs"):
    """Carrega todo `history.json` sob `root`, ordenado por algoritmo/variante/seed."""
    achados = []
    for base, _, arquivos in os.walk(root):
        for nome in arquivos:
            if nome == "history.json":
                achados.append(load(os.path.join(base, nome)))
    achados.sort(key=lambda r: (r.algo, r.variant, r.seed))
    return achados


# ------------------------------------------------------------------- validação
def validate(record: RunRecord, strict_eval=True):
    """Devolve a lista de violações do contrato. Lista vazia = pode entrar na arena.

    Curvas marcadas `comparable=False` só precisam ter identidade, curva e um `caveat`
    explicando por que não competem — o resto do contrato não se aplica a elas.
    """
    p = []

    if not record.algo:
        p.append("`algo` vazio")
    if record.schema_version != SCHEMA_VERSION:
        p.append(f"schema_version {record.schema_version} != {SCHEMA_VERSION}")
    if not record.curve:
        p.append("curva vazia")
    else:
        steps = [pt.get("global_step") for pt in record.curve]
        if any(s is None for s in steps):
            p.append("ponto da curva sem `global_step`")
        elif list(steps) != sorted(steps):
            p.append("`global_step` não é monotônico")

    if not record.comparable:
        if not record.caveat:
            p.append("`comparable=False` exige um `caveat` explicando por quê")
        return p

    # --- daqui para baixo, só para execuções que querem competir
    for chave, esperado in CONTRATO.items():
        obtido = record.env_spec.get(chave, "<ausente>")
        if obtido != esperado:
            p.append(f"env_spec['{chave}'] = {obtido!r}, contrato exige {esperado!r}")

    if not record.final:
        p.append("`final` vazio — falta o resultado do protocolo de avaliação")
    elif strict_eval:
        f = record.final
        if f.get("episodes") != CONTRATO["eval_episodes"]:
            p.append(f"avaliação final com {f.get('episodes')} episódios, "
                     f"contrato exige {CONTRATO['eval_episodes']}")
        if not f.get("completo", True):
            p.append("avaliação final incompleta (bateu `max_steps`)")
        media = f.get("score_mean")
        if media is None:
            p.append("`final.score_mean` ausente")
        elif not (0.0 <= media <= SCORE_PERFEITO):
            p.append(f"score_mean fora da faixa possível: {media}")

    orcamento = record.config.get("total_steps")
    if orcamento is None:
        p.append("`config['total_steps']` ausente — o orçamento é parte do contrato")
    elif int(orcamento) != ORCAMENTO_OFICIAL:
        p.append(f"orçamento de {int(orcamento):,} passos; o contrato exige "
                 f"{ORCAMENTO_OFICIAL:,}. Comparar treinos de tamanhos diferentes mede "
                 "paciência, não algoritmo")

    if record.params <= 0:
        p.append("`params` deve ser o número de parâmetros treináveis")
    if not record.net:
        p.append("`net` vazio — a arquitetura é um eixo de comparação")

    return p


def assert_valid(record: RunRecord, **kw):
    problemas = validate(record, **kw)
    if problemas:
        raise ContractViolation(
            f"{record.run_id} viola o contrato:\n  - " + "\n  - ".join(problemas)
        )
    return record


# ---------------------------------------------------------------------- legado
def from_legacy_csv(path, algo="dqn-legacy", variant=None, caveat=None):
    """Converte um CSV de treino do `colab-rl` para o esquema do repositório.

    Os CSVs antigos têm colunas sem nome: `índice, comprimento, passos, loss, reward`.
    O comprimento vira score pela regra `score = comprimento - 3`, e o registro nasce
    `comparable=False` — foi medido em outro ambiente, com outra recompensa e outra
    unidade de tempo. Ele é contexto histórico, não competidor.
    """
    import csv

    linhas = []
    with open(path, newline="", encoding="utf-8") as f:
        leitor = csv.reader(f)
        cabecalho = next(leitor, None)
        for row in leitor:
            if len(row) < 5:
                continue
            try:
                ep = int(float(row[0]))
                comprimento = float(row[1])
                passos = float(row[2])
                perda = float(row[3])
                recompensa = float(row[4])
            except ValueError:
                continue
            linhas.append((ep, comprimento, passos, perda, recompensa))

    if not linhas:
        raise ValueError(f"nenhuma linha aproveitável em {path} (cabeçalho: {cabecalho})")

    # Nos CSVs originais o nome do arquivo é sempre `keras_training_data.csv` e quem
    # identifica a variante é a pasta; nos normalizados de `results/legacy/` é o
    # contrário. Aceita os dois.
    if variant is None:
        raiz = os.path.splitext(os.path.basename(path))[0]
        variant = (os.path.basename(os.path.dirname(path))
                   if raiz in ("keras_training_data", "training_data") else raiz)
    curva = [
        {
            "global_step": ep,               # aqui o eixo é episódio, não passo — ver caveat
            "episodes": ep,
            "train_score_mean": comprimento - 3.0,
            "train_length_mean": comprimento,
            "episode_steps": passos,
            "loss": perda,
            "reward": recompensa,
        }
        for ep, comprimento, passos, perda, recompensa in linhas
    ]

    scores = np.array([c["train_score_mean"] for c in curva], dtype=np.float64)
    rec = RunRecord(
        algo=algo,
        variant=variant,
        seed=0,
        net="cnn-legado",
        params=0,
        env_spec={"env": "snake-on-pygame (legado)"},
        curve=curva,
        final={
            "episodes": len(curva),
            "score_mean": float(scores[-100:].mean()),
            "score_max": float(scores.max()),
        },
        comparable=False,
        caveat=(
            caveat
            or "Medido no ambiente antigo (snake-on-pygame): recompensa +comprimento ao "
               "comer, estado ordinal de 1 canal com a cabeça sobrescrita, 5 ações "
               "absolutas, eixo em episódios. Convertido por score = comprimento - 3 "
               "apenas para posicionar a curva; não é comparável com as execuções novas."
        ),
        meta={"fonte": os.path.basename(path)},
    )
    return rec


# ------------------------------------------------------------------ utilidades
def _jsonable(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, np.ndarray):
        return v.tolist()
    if isinstance(v, (np.bool_,)):
        return bool(v)
    return v


def _ambiente():
    """Carimbo de proveniência: sem isto, um número no gráfico não é rastreável."""
    meta = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    }
    try:
        meta["commit"] = subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL,
            text=True, timeout=5,
        ).strip()
    except Exception:
        meta["commit"] = "desconhecido"
    for mod in ("tensorflow", "keras"):
        try:
            meta[mod] = __import__(mod).__version__
        except Exception:
            pass
    return meta


# --- snakeai/env/render.py ---
"""Ver a cobra jogar — GIF de um episódio, sem pygame.

O Colab não tem display, então renderizar pelo jogo original não é opção. Aqui o episódio
vira uma sequência de imagens direto da grade `occ` do `VecSnake`, e o GIF é o artefato
que se olha para entender *como* o agente joga — coisa que nenhuma curva conta.

Vale mais do que parece: um agente com score médio 20 que morre sempre se prendendo no
próprio corpo e outro que morre por fome têm a mesma linha no gráfico e problemas
completamente diferentes.
"""


import numpy as np


__all__ = ["quadros_do_episodio", "render_episode", "PALETA_JOGO"]

#: Fundo, corpo, comida, cabeça — nas cores do gráfico da arena, para o GIF e as figuras
#: parecerem do mesmo projeto.
PALETA_JOGO = np.array(
    [
        [26, 26, 25],      # fundo (a superfície escura do gráfico)
        [27, 175, 122],    # corpo (aqua da paleta)
        [235, 104, 52],    # comida (laranja da paleta)
        [252, 252, 251],   # cabeça (tinta clara)
    ],
    dtype=np.uint8,
)


def _quadro(env: VecSnake, i=0, escala=16):
    grade = np.zeros((env.b, env.b), dtype=np.int32)
    grade[env.occ[i] > 0] = 1
    grade[env.food[i, 0], env.food[i, 1]] = 2
    grade[env.head[i, 0], env.head[i, 1]] = 3
    return PALETA_JOGO[grade].repeat(escala, 0).repeat(escala, 1)


def quadros_do_episodio(politica, board_size=10, safety=False, max_steps=2000,
                        seed=7, escala=16):
    """Roda um episódio com `politica` e devolve `(quadros, score, motivo)`.

    `politica` é a mesma interface de `snakeai.eval`: `politica(obs, mask) -> logits`.
    Assim o GIF mostra exatamente a política que o benchmark mediu, sem caminho paralelo.
    """

    env = VecSnake(1, board_size, rng=np.random.default_rng(seed))
    obs, mask = env.reset()
    quadros = [_quadro(env, escala=escala)]
    score, motivo = 0, "limite de passos"

    for _ in range(max_steps):
        logits = np.asarray(politica(obs, mask), dtype=np.float32)
        logits = np.where(mask, logits, MASK_NEG)
        if safety:
            logits = apply_safety_filter(env, logits)
        a = logits.argmax(axis=1).astype(np.int32)

        score_antes = int(env.score[0])
        comprimento_antes = int(env.length[0])
        fome_antes = int(env.hunger[0])
        obs, mask, r, d, info = env.step(a)
        quadros.append(_quadro(env, escala=escala))

        if d[0]:
            score = score_antes
            if comprimento_antes >= board_size * board_size - 1:
                motivo = "tabuleiro cheio"
            elif fome_antes + 1 >= env.starve_base + 2 * comprimento_antes:
                motivo = "fome"
            else:
                motivo = "colisão"
            break
    else:
        score = int(env.score[0])

    return quadros, score, motivo


def render_episode(politica, caminho="episodio.gif", fps=15, **kw):
    """Grava o GIF e devolve `(caminho, score, motivo)`."""
    import imageio.v2 as imageio

    quadros, score, motivo = quadros_do_episodio(politica, **kw)
    imageio.mimsave(caminho, quadros, fps=fps, loop=0)
    return caminho, score, motivo


# --- snakeai/export.py ---
"""Exportar o modelo — `.keras` para retomar treino, TFLite para embarcar.

Uma armadilha silenciosa do Keras 3, registrada aqui para ninguém repetir
--------------------------------------------------------------------------
Converter para TFLite **precisa passar por um SavedModel**.
`TFLiteConverter.from_concrete_functions(...)` compila sem erro, gera um arquivo
minúsculo — e **não captura os pesos**. A inferência devolve NaN, sem nenhum aviso. O
sintoma é um `.tflite` de poucos KB quando deveria ter centenas.

Por isso `export_model` sempre passa por `model.export(dir, format="tf_saved_model")`, e
sempre valida a paridade contra o modelo original antes de declarar sucesso.
"""


import os
import shutil
import time

import numpy as np


__all__ = ["export_model", "medir_latencia", "conferir_paridade"]


def medir_latencia(fn, board_size=10, repeticoes=200, aquecimento=20):
    """Latência de inferência com lote 1 — o que importa se o modelo for para o jogo."""
    x = np.zeros((1, board_size, board_size, N_CHANNELS), dtype=np.float32)
    for _ in range(aquecimento):
        fn(x)
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        fn(x)
    return (time.perf_counter() - t0) / repeticoes * 1000.0


def conferir_paridade(modelo, blob_tflite, board_size=10, n=200, seed=0):
    """O `.tflite` escolhe a mesma ação que o `.keras`, em `n` estados aleatórios?

    Não basta comparar os logits: o que importa para o jogo é a **ação escolhida**. Uma
    diferença numérica de quantização é aceitável; uma ação diferente não é.
    """
    import tensorflow as tf

    rng = np.random.default_rng(seed)
    x = rng.normal(size=(n, board_size, board_size, N_CHANNELS)).astype(np.float32)

    saida = modelo(x, training=False)
    logits_keras = np.asarray(saida[0] if isinstance(saida, (list, tuple)) else saida)

    itp = tf.lite.Interpreter(model_content=blob_tflite)
    itp.allocate_tensors()
    entrada = itp.get_input_details()[0]
    saidas = itp.get_output_details()

    logits_lite = []
    for i in range(n):
        itp.set_tensor(entrada["index"], x[i: i + 1])
        itp.invoke()
        cand = [itp.get_tensor(o["index"]) for o in saidas]
        # a saída de política é a que tem N_ACTIONS colunas
        pol = next((c for c in cand if c.shape[-1] == N_ACTIONS), cand[0])
        logits_lite.append(pol[0])
    logits_lite = np.array(logits_lite)

    if logits_keras.ndim == 3:      # C51: colapsa átomos só para comparar a escolha
        logits_keras = logits_keras.mean(-1)
    iguais = (logits_keras.argmax(1) == logits_lite.argmax(1)).mean()
    return {
        "acoes_iguais": float(iguais),
        "erro_max_logits": float(np.abs(logits_keras - logits_lite).max()),
    }


def export_model(modelo, out_dir="export", board_size=10, formatos=("fp16", "int8"),
                 validar=True):
    """Exporta e **mede**: tamanho, latência e paridade de ação.

    Devolve um dicionário pronto para virar linha do `MODELS.md`.
    """
    import tensorflow as tf

    os.makedirs(out_dir, exist_ok=True)
    caminho_keras = os.path.join(out_dir, "modelo.keras")
    modelo.save(caminho_keras)

    resultado = {
        "params": int(modelo.count_params()),
        "keras_kb": round(os.path.getsize(caminho_keras) / 1024, 1),
        "tf_ms": round(medir_latencia(lambda x: modelo(x, training=False), board_size), 4),
    }

    sm_dir = os.path.join(out_dir, "saved_model")
    if os.path.isdir(sm_dir):
        shutil.rmtree(sm_dir)
    modelo.export(sm_dir, format="tf_saved_model")

    for nome in formatos:
        conv = tf.lite.TFLiteConverter.from_saved_model(sm_dir)
        if nome != "fp32":
            conv.optimizations = [tf.lite.Optimize.DEFAULT]
        if nome == "fp16":
            conv.target_spec.supported_types = [tf.float16]
        blob = conv.convert()

        caminho = os.path.join(out_dir, f"modelo_{nome}.tflite")
        with open(caminho, "wb") as f:
            f.write(blob)
        resultado[f"{nome}_kb"] = round(len(blob) / 1024, 1)

        itp = tf.lite.Interpreter(model_content=blob)
        itp.allocate_tensors()
        ent = itp.get_input_details()[0]
        xi = np.zeros(ent["shape"], dtype=np.float32)

        def roda(_x, _itp=itp, _ent=ent):
            _itp.set_tensor(_ent["index"], xi)
            _itp.invoke()

        resultado[f"{nome}_ms"] = round(medir_latencia(roda, board_size), 4)

        if validar:
            resultado[f"{nome}_paridade"] = conferir_paridade(modelo, blob, board_size)
            if resultado[f"{nome}_kb"] < resultado["params"] / 4096:
                resultado[f"{nome}_alerta"] = (
                    "arquivo pequeno demais para esse número de parâmetros — "
                    "provável perda de pesos na conversão"
                )

    return resultado


# --- snakeai/plot.py ---
"""O gráfico da arena — onde os algoritmos finalmente ficam lado a lado.

Regras de leitura que este módulo impõe, e o porquê de cada uma:

* **Um eixo só.** Score de avaliação contra passos de ambiente. Nada de segundo eixo y:
  duas escalas empilhadas inventam correlação que não existe nos dados.
* **Cor é identidade, não posição.** Cada algoritmo recebe um slot fixo da paleta, sempre
  o mesmo. Filtrar a arena não repinta os sobreviventes — quem aprendeu que "PPO é azul"
  continua certo no gráfico seguinte.
* **Mediana com faixa interquartil**, nunca uma semente só. Uma curva de RL de execução
  única não é resultado, é anedota.
* **Curvas legadas em painel próprio.** Elas vêm de `comparable=False` e são medidas em
  *episódios*, não em passos de ambiente. Plotá-las no mesmo eixo x seria fabricar um eixo
  comum que não existe — o mesmo pecado do gráfico de dois eixos y, com outra roupa. Elas
  ganham um painel ao lado, com o próprio eixo rotulado, em cinza tracejado.
* **Piso e teto sempre visíveis.** Sem o piso aleatório de 1,21 desenhado, qualquer curva
  parece aprendizado; com ele, dá para ver quem só está tendo sorte.
* **Rótulo direto no fim de cada curva**, além da legenda. Três das cores da paleta clara
  ficam abaixo de 3:1 de contraste com o fundo, e a regra é que nesse caso a identidade
  não pode depender só da cor.

A paleta é a de referência do sistema de dataviz, validada para daltonismo nos dois modos
(pior par adjacente ΔE 9,1 no claro e 8,4 no escuro).
"""


import numpy as np

__all__ = ["PALETA", "cores_por_algoritmo", "arena_figure", "arena_table", "plot_run"]

# ---------------------------------------------------------------------- paleta
PALETA = {
    "light": {
        "surface": "#fcfcfb",
        "plane": "#f9f9f7",
        "ink": "#0b0b0b",
        "ink2": "#52514e",
        "muted": "#898781",
        "grid": "#e1e0d9",
        "axis": "#c3c2b7",
        "series": ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4",
                   "#008300", "#4a3aa7", "#e34948"],
        "legado": "#898781",
    },
    "dark": {
        "surface": "#1a1a19",
        "plane": "#0d0d0d",
        "ink": "#ffffff",
        "ink2": "#c3c2b7",
        "muted": "#898781",
        "grid": "#2c2c2a",
        "axis": "#383835",
        "series": ["#3987e5", "#d95926", "#199e70", "#c98500", "#d55181",
                   "#008300", "#9085e9", "#e66767"],
        "legado": "#898781",
    },
}

#: Ordem fixa dos slots. Um algoritmo novo entra no fim; ninguém troca de cor por isso.
ORDEM_ALGORITMOS = ["ppo", "dqn", "rainbow", "a2c", "acer", "alphazero",
                    "muzero", "acktr", "dreamerv3", "dqn-legacy"]

#: Famílias, na ordem em que os painéis aparecem. Existem porque a arena passou de oito
#: algoritmos e **oito é o limite honesto de uma paleta categórica**: a nona cor seria
#: indistinguível de alguma das oito sob daltonismo. A saída não é gerar mais uma cor, é
#: mudar a forma do gráfico — *small multiples*, um painel por família.
#:
#: O agrupamento é o de sempre em RL, e não uma conveniência visual: o que o algoritmo
#: aprende (política, valor, ou um modelo do mundo) é a divisão que explica por que as
#: curvas têm formatos diferentes.
FAMILIAS = [
    ("política", "gradiente de política", ["ppo", "a2c", "acktr", "acer"]),
    ("valor", "função de valor", ["dqn", "rainbow"]),
    ("modelo", "modelo do mundo e busca", ["alphazero", "muzero", "dreamerv3"]),
]


def familia_de(algo):
    for chave, _, membros in FAMILIAS:
        if algo in membros:
            return chave
    return "outros"

PISO_ALEATORIO = 1.21
SCORE_PERFEITO = 97


def cores_por_algoritmo(algoritmos, mode="light"):
    """Mapeia algoritmo -> cor, em ordem fixa. Nunca cicla nem gera hue nova.

    Passar do oitavo algoritmo é um erro deliberado: a nona cor seria
    indistinguível de alguma das oito sob daltonismo. Nesse ponto o gráfico
    precisa virar *small multiples*, não ganhar mais uma cor.
    """
    p = PALETA[mode]["series"]
    conhecidos = [a for a in ORDEM_ALGORITMOS if a in algoritmos]
    novos = sorted(a for a in algoritmos if a not in ORDEM_ALGORITMOS)
    ordenados = conhecidos + novos
    if len(ordenados) > len(p):
        raise ValueError(
            f"{len(ordenados)} algoritmos para {len(p)} slots de cor. "
            "Use `arena_figure(..., familias=True)` — small multiples por família — "
            "ou agrupe a cauda em 'outros'. Não gere cor nova."
        )
    return {a: p[i] for i, a in enumerate(ordenados)}


def cores_por_familia(mode="light"):
    """Cor de cada algoritmo **dentro do painel da sua família**.

    Nos *small multiples*, cada painel é uma unidade de leitura com no máximo quatro
    séries coloridas; as outras famílias aparecem em cinza, só para dar contexto. Duas
    famílias podem repetir um matiz — o que é seguro porque elas nunca aparecem coloridas
    no mesmo painel, e cada curva colorida ganha rótulo direto.

    A cor é presa ao algoritmo pela posição dele dentro da família, que é fixa. Filtrar
    execuções não repinta ninguém.
    """
    p = PALETA[mode]["series"]
    return {a: p[i] for _, _, membros in FAMILIAS for i, a in enumerate(membros)}


# ------------------------------------------------------------------ agregação
def agrega_sementes(registros, pontos=60):
    """Junta as sementes de uma mesma `(algo, variante)` numa mediana com faixa IQR.

    As sementes raramente avaliam nos mesmos passos, então interpolamos todas numa
    grade log-espaçada comum antes de tirar os quantis. A grade para no menor
    `max(step)` entre as sementes — extrapolar seria inventar dado.
    """
    curvas = []
    for r in registros:
        x, y = r.eval_curve()
        if x.size >= 2:
            curvas.append((x, y))
    if not curvas:
        return None

    x_min = max(1, max(c[0][0] for c in curvas))
    x_max = min(c[0][-1] for c in curvas)
    if x_max <= x_min:
        return None

    grade = np.unique(np.geomspace(x_min, x_max, pontos).astype(np.int64))
    empilhado = np.stack([np.interp(grade, x, y) for x, y in curvas])
    return {
        "x": grade,
        "mediana": np.median(empilhado, axis=0),
        "q1": np.percentile(empilhado, 25, axis=0),
        "q3": np.percentile(empilhado, 75, axis=0),
        "n_sementes": len(curvas),
    }


def _agrupa(registros):
    grupos = {}
    for r in registros:
        grupos.setdefault((r.algo, r.variant), []).append(r)
    return grupos


# -------------------------------------------------------------------- figuras
def arena_familias(registros, mode="light", figsize=(14.5, 4.8), titulo=None,
                   x_log=True, mostrar_legado=True):
    """*Small multiples*: um painel por família, com as demais em cinza ao fundo.

    Esta é a forma que a arena assume quando passa de oito algoritmos. Ela não é um
    consolo por não caber tudo num painel — é melhor para a pergunta que a arena de fato
    responde. Sobrepor nove curvas com faixa interquartil produz um emaranhado onde a
    comparação relevante ("o Rainbow supera o DQN?") fica *mais* difícil, não menos.

    Cada painel mostra a família em cor e **todas as outras curvas em cinza claro**, na
    mesma escala. Sem esse fundo, três painéis lado a lado seriam três gráficos
    independentes e a comparação entre famílias se perderia — que é justamente o que a
    arena existe para permitir.
    """
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter

    p = PALETA[mode]
    cores = cores_por_familia(mode)
    comparaveis = [r for r in registros if r.oficial]

    agregados = {}
    for (algo, variante), rs in sorted(_agrupa(comparaveis).items()):
        ag = agrega_sementes(rs)
        if ag is not None:
            agregados[(algo, variante)] = ag

    presentes = [f for f in FAMILIAS
                 if any(a in f[2] for a, _ in agregados)] or FAMILIAS
    legado = [r for r in registros if not r.comparable] if mostrar_legado else []

    # O painel legado entra com largura menor e **eixo x próprio**: ele mede episódios, e
    # pendurá-lo no eixo de passos seria fabricar um eixo comum que não existe.
    larguras = [1.0] * len(presentes) + ([0.62] if legado else [])
    fig = plt.figure(figsize=figsize, facecolor=p["plane"])
    gs = fig.add_gridspec(1, len(larguras), width_ratios=larguras, wspace=.08)
    axes = [fig.add_subplot(gs[0])]
    axes += [fig.add_subplot(gs[i], sharex=axes[0], sharey=axes[0])
             for i in range(1, len(presentes))]
    ax_leg = fig.add_subplot(gs[-1], sharey=axes[0]) if legado else None

    topo = max((max(ag["mediana"]) for ag in agregados.values()), default=0.0)
    topo = max(topo * 1.3, PISO_ALEATORIO * 4)

    for i, (ax, (chave, rotulo, membros)) in enumerate(zip(axes, presentes)):
        ax.set_facecolor(p["surface"])
        ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0, zorder=1)
        if i == len(presentes) - 1:
            # rotulada uma vez, e no painel mais vazio: repetir a referência nos três
            # seria ruído, e à esquerda ela cai em cima do início das curvas
            ax.annotate(f"piso aleatório · {PISO_ALEATORIO:.2f}".replace(".", ","),
                        xy=(0.98, PISO_ALEATORIO), xycoords=("axes fraction", "data"),
                        xytext=(0, 5), textcoords="offset points",
                        color=p["muted"], fontsize=8, va="bottom", ha="right")

        # contexto: todo o resto, em cinza, atrás
        for (algo, _), ag in agregados.items():
            if algo not in membros:
                ax.plot(ag["x"], ag["mediana"], color=p["legado"], lw=1.2,
                        alpha=.45, zorder=2, solid_capstyle="round")

        rotulos = []
        for (algo, variante), ag in agregados.items():
            if algo not in membros:
                continue
            cor = cores[algo]
            nome = algo if variante in ("default", "") else f"{algo} · {variante}"
            ax.fill_between(ag["x"], ag["q1"], ag["q3"], color=cor, alpha=.16,
                            linewidth=0, zorder=3)
            ax.plot(ag["x"], ag["mediana"], color=cor, lw=2.0, zorder=4,
                    label=f"{nome}  (n={ag['n_sementes']})", solid_capstyle="round")
            rotulos.append((ag["x"][-1], ag["mediana"][-1], nome, cor))

        for x, y, nome, _ in _sem_colisao(rotulos):
            ax.annotate(nome, xy=(x, y), xytext=(5, 0), textcoords="offset points",
                        color=p["ink2"], fontsize=8.5, va="center", ha="left", zorder=5)

        ax.set_title(rotulo, color=p["ink2"], fontsize=10.5, loc="left", pad=10)
        if x_log:
            ax.set_xscale("log")
        ax.set_ylim(0, topo)
        if not agregados:
            ax.set_xlim(1e4, 1e7)
        else:
            # espaço à direita para o rótulo direto de cada curva não sair do painel
            ax.margins(x=.30)
        ax.grid(True, which="major", color=p["grid"], lw=0.8, zorder=0)
        ax.set_axisbelow(True)
        for lado in ("top", "right"):
            ax.spines[lado].set_visible(False)
        for lado in ("left", "bottom"):
            ax.spines[lado].set_color(p["axis"])
        ax.tick_params(colors=p["muted"], labelsize=9, length=0)
        ax.xaxis.set_major_formatter(FuncFormatter(_formata_passos))
        ax.set_xlabel("passos de ambiente", color=p["ink2"], fontsize=9.5)
        if i:
            ax.tick_params(labelleft=False)
        if rotulos:
            leg = ax.legend(loc="upper left", frameon=False, fontsize=8.5,
                            labelcolor=p["ink2"], handlelength=1.6)
            for t in leg.get_texts():
                t.set_color(p["ink2"])

    if ax_leg is not None:
        _painel_legado(ax_leg, legado, p, ylim=(0, topo))
        ax_leg.tick_params(labelleft=False)

    axes[0].set_ylabel("score na avaliação (1.000 episódios, greedy)",
                       color=p["ink2"], fontsize=10)
    fig.suptitle(titulo or "snake-arena · por família de algoritmo",
                 color=p["ink"], fontsize=13, x=.006, ha="left", y=.985)
    fig.text(.006, .015,
             "cada painel colore uma família e mantém as demais curvas em cinza, na mesma "
             "escala; nove algoritmos não cabem numa paleta categórica. O painel do legado "
             "tem eixo x próprio, em episódios — ver docs/COMPARABILITY.md.",
             color=p["muted"], fontsize=8)
    fig.subplots_adjust(left=.058, right=.985, top=.83, bottom=.16)
    return fig, tuple(axes) + ((ax_leg,) if ax_leg is not None else ())


def arena_figure(registros, mode="light", figsize=(12.5, 6.2), titulo=None,
                 mostrar_legado=True, x_log=True, familias="auto"):
    """A figura principal do benchmark. Devolve `(fig, (ax, ax_legado))`.

    `familias="auto"` (o padrão) troca para *small multiples* assim que o número de
    algoritmos passa dos oito slots de cor. É automático de propósito: a alternativa
    seria a arena quebrar — ou, pior, ganhar uma nona cor — no dia em que o nono
    algoritmo termina de treinar.

    `registros` é uma lista de `snakeai.record.RunRecord` — tipicamente
    `record.load_all("runs")` mais as curvas legadas convertidas.

    O painel grande tem só as execuções `comparable=True`, no eixo oficial de passos de
    ambiente. As legadas, quando existem, vão para um painel estreito à direita com o
    **próprio eixo em episódios** — porque é isso que elas medem, e fingir o contrário
    seria exatamente o erro que este repositório foi criado para consertar.
    """
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter

    p = PALETA[mode]
    comparaveis = [r for r in registros if r.oficial]
    legado = [r for r in registros if not r.comparable] if mostrar_legado else []

    algos = {r.algo for r in comparaveis}
    if familias is True or (familias == "auto" and len(algos) > len(p["series"])):
        return arena_familias(registros, mode=mode, titulo=titulo, x_log=x_log,
                              mostrar_legado=mostrar_legado)

    cores = cores_por_algoritmo(algos, mode)

    fig = plt.figure(figsize=figsize, facecolor=p["plane"])
    if legado:
        gs = fig.add_gridspec(1, 2, width_ratios=(3.4, 1), wspace=.22)
        ax = fig.add_subplot(gs[0])
        ax_leg = fig.add_subplot(gs[1])
    else:
        ax = fig.add_subplot(1, 1, 1)
        ax_leg = None
    ax.set_facecolor(p["surface"])

    # --- referências primeiro, para ficarem atrás dos dados
    ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0, zorder=1)
    ax.annotate(f"piso aleatório com máscara · {PISO_ALEATORIO:.2f}".replace(".", ","),
                xy=(0.995, PISO_ALEATORIO), xycoords=("axes fraction", "data"),
                xytext=(0, 5), textcoords="offset points",
                color=p["muted"], fontsize=8.5, va="bottom", ha="right")

    # --- as curvas que competem
    rotulos = []
    for (algo, variante), rs in sorted(_agrupa(comparaveis).items()):
        ag = agrega_sementes(rs)
        if ag is None:
            continue
        cor = cores[algo]
        nome = algo if variante in ("default", "") else f"{algo} · {variante}"
        ax.fill_between(ag["x"], ag["q1"], ag["q3"], color=cor, alpha=.16,
                        linewidth=0, zorder=3)
        ax.plot(ag["x"], ag["mediana"], color=cor, lw=2.0, zorder=4,
                label=f"{nome}  (n={ag['n_sementes']})", solid_capstyle="round")
        rotulos.append((ag["x"][-1], ag["mediana"][-1], nome, cor))

    # --- rótulo direto no fim de cada curva (a "relief rule" do contraste)
    for x, y, nome, cor in _sem_colisao(rotulos):
        ax.annotate(nome, xy=(x, y), xytext=(6, 0), textcoords="offset points",
                    color=p["ink2"], fontsize=9, va="center", ha="left", zorder=5)

    # --- eixos e cromo
    if x_log:
        ax.set_xscale("log")
    ax.set_xlabel("passos de ambiente", color=p["ink2"], fontsize=10)
    ax.set_ylabel("score na avaliação (1.000 episódios, greedy)",
                  color=p["ink2"], fontsize=10)
    ax.set_title(titulo or "snake-arena · mesmo ambiente, mesmo orçamento, mesma régua",
                 color=p["ink"], fontsize=13, pad=14, loc="left")

    ax.grid(True, which="major", color=p["grid"], lw=0.8, ls="-", zorder=0)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(p["axis"])
        ax.spines[lado].set_linewidth(1.0)
    ax.tick_params(colors=p["muted"], labelsize=9, length=0)
    ax.xaxis.set_major_formatter(FuncFormatter(_formata_passos))

    # o teto do eixo y vem de TODOS os dados, inclusive os legados: os dois painéis
    # compartilham a escala de score, e calcular só a partir das curvas oficiais faz o
    # painel da direita ser cortado quando a arena ainda está vazia
    topo_oficial = max((y for _, y, _, _ in rotulos), default=0.0)
    topo_legado = max(
        (max(c["train_score_mean"] for c in r.curve) for r in legado), default=0.0
    ) if legado else 0.0
    topo = max(topo_oficial * 1.3, topo_legado * 1.15, PISO_ALEATORIO * 4)
    ax.set_ylim(0, topo)
    if rotulos:
        ax.margins(x=.18)
    else:
        # arena vazia: um eixo x de 1 a 10 e um retângulo em branco não comunicam nada
        ax.set_xlim(1e4, 1e7)
        ax.annotate(
            "nenhuma execução oficial ainda\n\n"
            "as curvas entram aqui quando forem treinadas no orçamento do contrato",
            xy=(0.5, 0.55), xycoords="axes fraction", ha="center", va="center",
            color=p["muted"], fontsize=11, linespacing=1.6)

    if len(rotulos) >= 2:
        leg = ax.legend(loc="upper left", frameon=False, fontsize=9,
                        labelcolor=p["ink2"], handlelength=1.6)
        for t in leg.get_texts():
            t.set_color(p["ink2"])

    # --- painel legado: eixo próprio, unidade própria, mesma escala de score
    if ax_leg is not None:
        _painel_legado(ax_leg, legado, p, ylim=ax.get_ylim())
        fig.text(0.012, 0.015,
                 "Os dois painéis não compartilham eixo x — e não podem. À esquerda, "
                 "passos de ambiente no jogo novo; à direita, episódios no jogo de 2019, "
                 "com outra recompensa e score de treino em vez de avaliação.",
                 color=p["muted"], fontsize=8)
        fig.subplots_adjust(left=.075, right=.985, top=.88, bottom=.135)
    else:
        fig.tight_layout()
    return fig, (ax, ax_leg)


def _painel_legado(ax, legado, p, ylim=None):
    """As curvas históricas, no eixo delas: episódios de treino.

    Compartilham a escala y com o painel principal — score é score, essa parte é
    conversível. O eixo x é que não é, e por isso está separado.
    """
    ax.set_facecolor(p["surface"])
    ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0, zorder=1)

    melhor = (None, -1.0)
    for r in legado:
        x = np.array([c["episodes"] for c in r.curve], dtype=np.float64)
        y = np.array([c["train_score_mean"] for c in r.curve], dtype=np.float64)
        if y.size > 400:
            # 10 mil episódios num painel estreito viram um borrão cinza; a janela
            # larga mostra a tendência, que é o que o painel de contexto precisa dizer.
            k = max(1, y.size // 40)
            nucleo = np.ones(k) / k
            y = np.convolve(y, nucleo, mode="valid")
            x = x[k - 1:]
        ax.plot(x, y, color=p["legado"], lw=1.2, ls=(0, (4, 3)), alpha=.6, zorder=2)
        if y.max() > melhor[1]:
            melhor = (r.variant, float(y.max()), float(x[int(y.argmax())]))

    if melhor[0]:
        # rótulo ancorado no canto, não no ponto: no painel estreito um rótulo junto
        # ao máximo sai pela borda direita
        ax.annotate(f"melhor: {melhor[0]}\nmédia móvel {melhor[1]:.1f}".replace(".", ","),
                    xy=(0.04, 0.97), xycoords="axes fraction",
                    color=p["ink2"], fontsize=8.5, ha="left", va="top",
                    linespacing=1.5)

    ax.set_title("legado · 2019", color=p["ink2"], fontsize=10, loc="left", pad=14)
    ax.set_xlabel("episódios de treino", color=p["muted"], fontsize=9)
    if ylim:
        ax.set_ylim(ylim)
    ax.grid(True, color=p["grid"], lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(p["axis"])
    ax.tick_params(colors=p["muted"], labelsize=8.5, length=0)
    ax.xaxis.set_major_formatter(__import__("matplotlib").ticker.FuncFormatter(_formata_passos))


def _sem_colisao(rotulos, minimo=0.045):
    """Empurra rótulos que ficariam sobrepostos, preservando a ordem vertical."""
    if not rotulos:
        return []
    ordenado = sorted(rotulos, key=lambda t: t[1])
    ys = [t[1] for t in ordenado]
    faixa = max(ys[-1] - ys[0], 1e-9)
    minimo = minimo * faixa
    for i in range(1, len(ys)):
        if ys[i] - ys[i - 1] < minimo:
            ys[i] = ys[i - 1] + minimo
    return [(x, ys[i], nome, cor) for i, (x, _, nome, cor) in enumerate(ordenado)]


def _formata_passos(v, _pos=None):
    if v >= 1e6:
        return f"{v / 1e6:g} M"
    if v >= 1e3:
        return f"{v / 1e3:g} mil"
    return f"{v:g}"


def plot_run(record, mode="light", figsize=(11, 3.4)):
    """Diagnóstico de uma execução: treino (com exploração) contra avaliação (honesta).

    As duas subindo juntas = aprendeu. A de treino subindo sozinha = está explorando com
    sorte, e o número honesto não acompanha.
    """
    import matplotlib.pyplot as plt

    p = PALETA[mode]
    fig, ax = plt.subplots(figsize=figsize, facecolor=p["plane"])
    ax.set_facecolor(p["surface"])

    treino = [(c["global_step"], c["train_score_mean"]) for c in record.curve
              if c.get("train_score_mean") is not None]
    if treino:
        x, y = zip(*treino)
        ax.plot(x, y, color=p["muted"], lw=1.4, label="treino (com exploração)")

    x, y = record.eval_curve()
    if x.size:
        ax.plot(x, y, color=p["series"][0], lw=2.0, label="avaliação (greedy)")

    ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0)
    ax.set_xlabel("passos de ambiente", color=p["ink2"], fontsize=10)
    ax.set_ylabel("score", color=p["ink2"], fontsize=10)
    ax.set_title(record.run_id, color=p["ink"], fontsize=12, loc="left", pad=10)
    ax.grid(True, color=p["grid"], lw=0.8)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(p["axis"])
    ax.tick_params(colors=p["muted"], labelsize=9, length=0)
    leg = ax.legend(frameon=False, fontsize=9)
    for t in leg.get_texts():
        t.set_color(p["ink2"])
    fig.tight_layout()
    return fig, ax


# --------------------------------------------------------------------- tabela
def arena_table(registros, markdown=True):
    """A tabela de resultados — a visão que o gráfico não dá.

    Existe também porque três cores da paleta clara ficam abaixo de 3:1 de contraste:
    a regra manda oferecer rótulos visíveis **ou** a visão em tabela. Aqui temos as duas.
    """
    linhas = []
    for (algo, variante), rs in sorted(_agrupa([r for r in registros if r.oficial]).items()):
        finais = [r.final for r in rs if r.final]
        if not finais:
            continue
        medias = np.array([f["score_mean"] for f in finais], dtype=np.float64)
        passos = max((r.curve[-1]["global_step"] for r in rs if r.curve), default=0)
        linhas.append({
            "algo": algo,
            "variante": variante,
            "rede": rs[0].net,
            "params": rs[0].params,
            "sementes": len(rs),
            "passos": int(passos),
            "score_mean": float(np.median(medias)),
            "score_spread": float(medias.max() - medias.min()) if len(medias) > 1 else 0.0,
            "score_median": float(np.median([f.get("score_median", np.nan) for f in finais])),
            "score_max": int(max(f.get("score_max", 0) for f in finais)),
            "win_rate": float(np.median([f.get("win_rate", 0.0) for f in finais])),
        })
    linhas.sort(key=lambda d: -d["score_mean"])

    if not markdown:
        return linhas

    out = [
        "| algoritmo | rede | params | sementes | passos | score médio | amplitude | mediana | máx | cheio |",
        "|---|---|---|---:|---:|---:|---:|---:|---:|---:|",
        f"| _piso aleatório_ | — | — | — | 0 | **{PISO_ALEATORIO:.2f}** | — | 1 | — | 0% |".replace(".", ","),
    ]
    for d in linhas:
        nome = d["algo"] if d["variante"] in ("default", "") else f"{d['algo']} · {d['variante']}"
        out.append(
            f"| {nome} | `{d['rede']}` | {d['params']:,} | {d['sementes']} | "
            f"{d['passos']:,} | **{d['score_mean']:.2f}** | ±{d['score_spread']:.2f} | "
            f"{d['score_median']:.0f} | {d['score_max']} | {d['win_rate']:.1%} |"
        )
    out.append(f"\nScore perfeito no 10×10: **{SCORE_PERFEITO}**.")
    return "\n".join(out)


# --- snakeai/nets/resnet.py ---
"""Tronco residual totalmente convolucional — a rede do PPO.

No espírito do AlphaZero, mas minúsculo. Convoluções 3×3 com `padding="same"` num
tabuleiro 10×10 dão campo receptivo global depois de ~5 camadas, então 3 blocos residuais
já enxergam o tabuleiro inteiro — **sem jogar fora a posição**, que é onde as redes com
pooling do repositório antigo se perdiam.

Sobre normalização: PPO e BatchNorm se dão mal. As estatísticas do rollout não batem com
as do minibatch de update, e o valor aprendido fica dependente do tamanho do lote. Usamos
**GroupNorm**, que normaliza por amostra e não tem esse problema — e que funciona igual
para DQN, o que mantém a comparação limpa.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

from keras import layers

__all__ = ["PRESETS", "residual_block", "resnet", "TRONCOS_RESIDUAIS"]

#: nome -> (largura, número de blocos residuais)
PRESETS = {
    "resnet_tiny": (32, 2),     # ~40k params com a cabeça
    "resnet_small": (48, 3),    # ~135k — o ponto doce
    "resnet_base": (64, 4),     # ~320k
}


def residual_block(x, largura, nome):
    y = layers.Conv2D(largura, 3, padding="same", use_bias=False,
                      kernel_initializer="he_normal", name=f"{nome}_c1")(x)
    y = layers.GroupNormalization(groups=8, name=f"{nome}_n1")(y)
    y = layers.Activation("relu", name=f"{nome}_a1")(y)
    y = layers.Conv2D(largura, 3, padding="same", use_bias=False,
                      kernel_initializer="he_normal", name=f"{nome}_c2")(y)
    y = layers.GroupNormalization(groups=8, name=f"{nome}_n2")(y)
    out = layers.Add(name=f"{nome}_add")([x, y])
    return layers.Activation("relu", name=f"{nome}_a2")(out)


def resnet(x, preset="resnet_small", nome=None):
    """Tronco residual. Devolve o mapa de features `(B, B, largura)`, **sem achatar**.

    Não achatar é de propósito: as cabeças convolucionais 1×1 de `heads.py` aproveitam a
    estrutura espacial, e achatar cedo seria desperdiçá-la.
    """
    if preset not in PRESETS:
        raise ValueError(f"preset desconhecido: {preset!r}. Use um de {list(PRESETS)}")
    largura, blocos = PRESETS[preset]
    nome = nome or preset

    x = layers.Conv2D(largura, 3, padding="same", use_bias=False,
                      kernel_initializer="he_normal", name=f"{nome}_stem_c")(x)
    x = layers.GroupNormalization(groups=8, name=f"{nome}_stem_n")(x)
    x = layers.Activation("relu", name=f"{nome}_stem_a")(x)
    for i in range(blocos):
        x = residual_block(x, largura, f"{nome}_res{i}")
    return x


TRONCOS_RESIDUAIS = {
    nome: (lambda x, _p=nome: resnet(x, preset=_p)) for nome in PRESETS
}


# --- snakeai/nets/classic.py ---
"""Os troncos convolucionais do `colab-rl`, portados para Keras 3 e corrigidos.

Estes são os corpos de rede que produziram as curvas históricas. Estão aqui para que a
pergunta "quanto do ganho é o algoritmo e quanto é a arquitetura?" tenha resposta medida
em vez de opinião.

Duas coisas que a portabilidade revelou, e que valem mais que o código
------------------------------------------------------------------------

**1. "CNN2" significava duas coisas diferentes no mesmo repositório.**

O `colab-rl` tinha as CNNs definidas em dois lugares, com a mesma numeração e conteúdo
diferente:

===========  ==============================  ==================================
nome         em `models/utilities/networks.py`  nos notebooks
===========  ==============================  ==================================
``CNN1``     16→32, **quebrada** (`return model`)  32→64→64 (Rainbow)
``CNN2``     16→32→32, **quebrada**             32→64→64 com regularização L2
``CNN3``     32→64→64 (Rainbow)                 3 blocos VGG com max-pooling
``CNN4``     não existia                        idem CNN3, com dropout
===========  ==============================  ==================================

Ou seja: o notebook chamado *"DQN (RMSprop - CNN2 - KL-Divergence)"* usava um tronco que
**não é** a `CNN2` do pacote. Um leitor que fosse ao `networks.py` entender o experimento
leria a rede errada. Aqui as redes têm nome descritivo (`cnn_rainbow`, `cnn_alphazero`,
`cnn_vgg`, `cnn_vgg_dropout`) e os apelidos numéricos apontam para as definições **dos
notebooks**, que são as que de fato rodaram.

**2. As redes com pooling destroem o tabuleiro.**

`cnn_vgg` e `cnn_vgg_dropout` aplicam três `MaxPooling2D(2, 2)` seguidos. Num tabuleiro
10×10 isso é ``10 → 5 → 2 → 1``: a saída do tronco tem **uma única célula**. Toda a
informação de *onde* as coisas estão no tabuleiro é jogada fora antes da cabeça densa —
sobra só "existe corpo em algum lugar", "existe comida em algum lugar".

Essas arquiteturas foram desenhadas para imagens 224×224, onde três poolings deixam 28×28.
Copiadas para 10×10, elas colapsam. É uma explicação forte para o platô dos notebooks que
as usavam, e por isso `cnn_vgg_sem_pool` existe: mesma rede, sem os poolings, para medir
exatamente quanto custou.

Os troncos são fiéis ao original de propósito. A correção fica na cabeça (`heads.py`) e
nas variantes explicitamente marcadas.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, regularizers

__all__ = [
    "cnn_rainbow",
    "cnn_alphazero",
    "cnn_vgg",
    "cnn_vgg_dropout",
    "cnn_vgg_sem_pool",
    "TRONCOS_CLASSICOS",
    "APELIDOS_LEGADOS",
]


def cnn_rainbow(x, nome="cnn_rainbow"):
    """32→64→64 com kernels 3×3, 2×2, 1×1, sem padding.

    Da implementação do Rainbow do @Kaixhin. É a `CNN1` dos notebooks e a `CNN3` do
    `networks.py` — a mesma rede com dois nomes.
    """
    x = layers.Conv2D(32, 3, activation="relu", name=f"{nome}_c1")(x)
    x = layers.Conv2D(64, 2, activation="relu", name=f"{nome}_c2")(x)
    x = layers.Conv2D(64, 1, activation="relu", name=f"{nome}_c3")(x)
    return layers.Flatten(name=f"{nome}_flat")(x)


def cnn_alphazero(x, l2const=1e-4, nome="cnn_alphazero"):
    """A mesma pilha da `cnn_rainbow`, com regularização L2 e ativação separada.

    É a `CNN2` **dos notebooks** — a que rodou no experimento "CNN2 - KL-Divergence".
    """
    reg = regularizers.l2(l2const)
    for i, (filtros, k) in enumerate(((32, 3), (64, 2), (64, 1)), start=1):
        x = layers.Conv2D(filtros, k, kernel_regularizer=reg, name=f"{nome}_c{i}")(x)
        x = layers.Activation("relu", name=f"{nome}_a{i}")(x)
    return layers.Flatten(name=f"{nome}_flat")(x)


def _blocos_vgg(x, nome, dropout=0.0, pooling=True):
    plano = ((16, 2), (32, 2), (64, 3))
    for b, (filtros, convs) in enumerate(plano, start=1):
        for c in range(1, convs + 1):
            x = layers.Conv2D(filtros, 3, activation="relu", padding="same",
                              name=f"{nome}_b{b}_c{c}")(x)
            if dropout:
                x = layers.Dropout(dropout, name=f"{nome}_b{b}_d{c}")(x)
        if pooling:
            x = layers.MaxPooling2D(2, strides=2, name=f"{nome}_b{b}_pool")(x)
    return layers.Flatten(name=f"{nome}_flat")(x)


def cnn_vgg(x, nome="cnn_vgg"):
    """Três blocos no estilo VGG com max-pooling. É a `CNN3` dos notebooks.

    **Atenção:** os três poolings reduzem um tabuleiro 10×10 a 1×1. Ver o cabeçalho do
    módulo. Mantida fiel ao original porque é o que produziu as curvas históricas.
    """
    return _blocos_vgg(x, nome, dropout=0.0, pooling=True)


def cnn_vgg_dropout(x, nome="cnn_vgg_dropout", taxa=0.1):
    """`cnn_vgg` com dropout de 0,1 após cada convolução. É a `CNN4` dos notebooks."""
    return _blocos_vgg(x, nome, dropout=taxa, pooling=True)


def cnn_vgg_sem_pool(x, nome="cnn_vgg_sem_pool"):
    """`cnn_vgg` sem os max-poolings — a variante de ablação.

    Não existia no repositório antigo. Existe aqui para responder, com número, quanto do
    platô daquelas execuções veio de colapsar o tabuleiro a uma célula.
    """
    return _blocos_vgg(x, nome, dropout=0.0, pooling=False)


#: Nome descritivo -> função de tronco.
TRONCOS_CLASSICOS = {
    "cnn_rainbow": cnn_rainbow,
    "cnn_alphazero": cnn_alphazero,
    "cnn_vgg": cnn_vgg,
    "cnn_vgg_dropout": cnn_vgg_dropout,
    "cnn_vgg_sem_pool": cnn_vgg_sem_pool,
}

#: Apelidos numéricos do repositório antigo. Apontam para as definições **dos
#: notebooks**, que são as que realmente rodaram (ver o cabeçalho do módulo).
APELIDOS_LEGADOS = {
    "cnn1": "cnn_rainbow",
    "cnn2": "cnn_alphazero",
    "cnn3": "cnn_vgg",
    "cnn4": "cnn_vgg_dropout",
}


# --- snakeai/nets/heads.py ---
"""Cabeças de rede — dueling, noisy e distribucional (C51).

São os componentes que separam um DQN simples de um Rainbow. Ficam separados dos troncos
de propósito: qualquer cabeça encaixa em qualquer tronco, e é isso que permite perguntar
"quanto o dueling vale?" com o resto do experimento congelado.

Todas foram reescritas para Keras 3. A `NoisyDense` do repositório antigo herdava de
`Dense` e mexia nos internals dela (`self.kernel`, `build` reimplementado), o que quebra
em qualquer versão moderna; esta é uma `Layer` própria, com `add_weight` e `keras.random`.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

__all__ = ["NoisyDense", "dueling_head", "distributional_head", "q_de_distribuicao"]


@keras.saving.register_keras_serializable(package="snakeai")
class NoisyDense(layers.Layer):
    """Camada densa com ruído fatorado nos pesos (Fortunato et al., 2017).

    Substitui a exploração ε-greedy por ruído aprendido: a rede começa barulhenta e vai
    reduzindo o próprio σ conforme fica confiante. A vantagem sobre o ε-greedy é que a
    exploração passa a ser **dependente do estado** — o agente explora onde ainda não sabe,
    não uniformemente.

    Uma decisão importante: **o ruído é desligado quando `training=False`**. O protocolo de
    avaliação do contrato é greedy e determinístico; se a rede sorteasse ruído durante o
    benchmark, o mesmo modelo daria números diferentes a cada execução e a comparação entre
    algoritmos perderia o sentido. Alguns trabalhos mantêm o ruído na avaliação — aqui não,
    e a escolha está registrada porque muda o número publicado.

    Parâmetros
    ----------
    units : int
        Dimensão de saída.
    sigma0 : float
        Escala inicial do ruído, dividida por `sqrt(entrada)`. 0,5 é o valor do paper.
    """

    def __init__(self, units, activation=None, sigma0=0.5, seed=None, **kw):
        super().__init__(**kw)
        self.units = int(units)
        self.activation = keras.activations.get(activation)
        self.sigma0 = float(sigma0)
        self.seed = seed
        self.seed_generator = keras.random.SeedGenerator(seed)

    def build(self, input_shape):
        entrada = int(input_shape[-1])
        limite = 1.0 / (entrada ** 0.5)
        sigma_ini = self.sigma0 / (entrada ** 0.5)

        self.w_mu = self.add_weight(
            shape=(entrada, self.units), name="w_mu",
            initializer=keras.initializers.RandomUniform(-limite, limite))
        self.w_sigma = self.add_weight(
            shape=(entrada, self.units), name="w_sigma",
            initializer=keras.initializers.Constant(sigma_ini))
        self.b_mu = self.add_weight(
            shape=(self.units,), name="b_mu",
            initializer=keras.initializers.RandomUniform(-limite, limite))
        self.b_sigma = self.add_weight(
            shape=(self.units,), name="b_sigma",
            initializer=keras.initializers.Constant(sigma_ini))
        self._entrada = entrada

    @staticmethod
    def _f(x):
        """`sign(x) * sqrt(|x|)` — a transformação que fatora o ruído no paper."""
        return ops.sign(x) * ops.sqrt(ops.abs(x))

    def call(self, inputs, training=False):
        if training:
            eps_in = self._f(keras.random.normal((self._entrada,),
                                                 seed=self.seed_generator))
            eps_out = self._f(keras.random.normal((self.units,),
                                                  seed=self.seed_generator))
            w = self.w_mu + self.w_sigma * ops.outer(eps_in, eps_out)
            b = self.b_mu + self.b_sigma * eps_out
        else:
            w, b = self.w_mu, self.b_mu

        y = ops.matmul(inputs, w) + b
        return self.activation(y) if self.activation is not None else y

    def compute_output_shape(self, input_shape):
        return (*input_shape[:-1], self.units)

    def ruido_medio(self):
        """σ médio dos pesos — cai conforme a rede fica confiante. Bom de registrar."""
        return float(ops.convert_to_numpy(ops.mean(ops.abs(self.w_sigma))))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "units": self.units,
            "activation": keras.activations.serialize(self.activation),
            "sigma0": self.sigma0,
            "seed": self.seed,
        })
        return cfg


def _densa(tipo, unidades, ativacao=None, nome=None):
    if tipo == "noisy":
        return NoisyDense(unidades, activation=ativacao, name=nome)
    return layers.Dense(unidades, activation=ativacao, name=nome)


def dueling_head(x, n_actions, largura=256, densa="dense", nome="dueling"):
    """`Q(s,a) = V(s) + A(s,a) − média_a A(s,a)`.

    A subtração da média é o que torna a decomposição identificável: sem ela, somar uma
    constante a `V` e subtraí-la de `A` daria o mesmo `Q`, e as duas correntes poderiam
    derivar sem que a perda percebesse.

    O original usava a **média**; o paper também oferece o **máximo**. Ficamos na média,
    que é o padrão do Rainbow.
    """
    a = _densa(densa, largura, "relu", f"{nome}_a_h")(x)
    a = _densa(densa, n_actions, None, f"{nome}_a")(a)
    v = _densa(densa, largura, "relu", f"{nome}_v_h")(x)
    v = _densa(densa, 1, None, f"{nome}_v")(v)

    a_centrada = layers.Lambda(
        lambda t: t - ops.mean(t, axis=-1, keepdims=True),
        output_shape=lambda s: s, name=f"{nome}_center",
    )(a)
    return layers.Add(name=f"{nome}_q")([v, a_centrada])


def distributional_head(x, n_actions, n_atoms=51, largura=256, densa="dense",
                        dueling=False, nome="c51"):
    """Cabeça categórica do C51: distribuição sobre `n_atoms` valores por ação.

    Em vez de estimar `Q(s,a)` — a média do retorno — o C51 estima a distribuição inteira.
    O ganho não é só estatístico: aprender uma distribuição dá um sinal de treino mais
    rico por transição, e é a peça que mais contribui no Rainbow.

    Devolve **logits** de forma `(lote, n_ações, n_átomos)`. A softmax e a projeção sobre
    o suporte ficam no agente, onde o `v_min`/`v_max` é conhecido.
    """
    if dueling:
        a = _densa(densa, largura, "relu", f"{nome}_a_h")(x)
        a = _densa(densa, n_actions * n_atoms, None, f"{nome}_a")(a)
        a = layers.Reshape((n_actions, n_atoms), name=f"{nome}_a_r")(a)

        v = _densa(densa, largura, "relu", f"{nome}_v_h")(x)
        v = _densa(densa, n_atoms, None, f"{nome}_v")(v)
        v = layers.Reshape((1, n_atoms), name=f"{nome}_v_r")(v)

        a_centrada = layers.Lambda(
            lambda t: t - ops.mean(t, axis=1, keepdims=True),
            output_shape=lambda s: s, name=f"{nome}_center",
        )(a)
        return layers.Add(name=f"{nome}_logits")([v, a_centrada])

    h = _densa(densa, largura, "relu", f"{nome}_h")(x)
    h = _densa(densa, n_actions * n_atoms, None, f"{nome}_d")(h)
    return layers.Reshape((n_actions, n_atoms), name=f"{nome}_logits")(h)


def q_de_distribuicao(logits, suporte):
    """Colapsa a distribuição categórica em `Q(s,a)` — só para escolher a ação.

    `logits`: `(lote, n_ações, n_átomos)`. `suporte`: `(n_átomos,)`.
    """
    p = ops.softmax(logits, axis=-1)
    return ops.sum(p * ops.reshape(suporte, (1, 1, -1)), axis=-1)


def suporte_c51(v_min=-10.0, v_max=10.0, n_atoms=51):
    """Os `n_atoms` valores igualmente espaçados em `[v_min, v_max]`.

    Com recompensa `+1`/`−1` e γ = 0,995, o retorno de um episódio de Snake fica bem
    dentro de `[−2, 60]` — a faixa padrão de `[−10, 10]` do Atari é estreita demais aqui.
    O agente escolhe a sua; este é só o utilitário.
    """
    import numpy as np

    return np.linspace(v_min, v_max, n_atoms, dtype=np.float32)


# --- snakeai/nets/registry.py ---
"""O registro de redes — qualquer tronco, para qualquer algoritmo, por string.

É isto que transforma "qual arquitetura é melhor?" numa ablação medida: o agente recebe
`net="cnn_vgg"` ou `net="resnet_small"` e o resto do experimento não muda. Sem isso, cada
comparação de rede viraria um notebook novo, que é como o repositório antigo acabou com
seis DQNs que ninguém conseguia comparar.

Todo modelo construído aqui obedece ao contrato: entrada `(B, B, 5)` egocêntrica,
saída de política com 3 ações relativas.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers


__all__ = [
    "TRONCOS",
    "listar_troncos",
    "build_backbone",
    "build_actor_critic",
    "build_q_network",
    "build_policy_q",
    "resumo",
]

TRONCOS = {**TRONCOS_RESIDUAIS, **TRONCOS_CLASSICOS}

#: A cabeça densa do repositório antigo tinha **3136** unidades. O número não é arbitrário
#: — é exatamente o achatamento da `cnn_rainbow` num tabuleiro 10×10 (7×7×64), o mesmo
#: valor do DQN do Atari por coincidência de kernels. Só que uma camada de 3136 sobre uma
#: entrada de 3136 são **9,8 milhões de parâmetros**, e ela era replicada nas duas
#: correntes do dueling. Sobre a `cnn_vgg`, que entrega 64 features, a mesma camada liga
#: 64 entradas a 3136 unidades: quase toda a capacidade do modelo depois de o tronco já
#: ter descartado a informação espacial.
#: O padrão aqui é 256. `LARGURA_DENSA_LEGADA` continua disponível para reproduzir o
#: original quando a fidelidade importar mais que o bom senso.
LARGURA_DENSA_LEGADA = 3136
LARGURA_DENSA_PADRAO = 256


def listar_troncos():
    """Nomes aceitos, incluindo os apelidos numéricos do repositório antigo."""
    return sorted(TRONCOS) + sorted(APELIDOS_LEGADOS)


def _resolve(nome):
    if nome in TRONCOS:
        return TRONCOS[nome], nome
    if nome in APELIDOS_LEGADOS:
        canonico = APELIDOS_LEGADOS[nome]
        return TRONCOS[canonico], canonico
    raise ValueError(
        f"tronco desconhecido: {nome!r}. Disponíveis: {listar_troncos()}"
    )


def build_backbone(entrada, net="resnet_small"):
    """Aplica o tronco `net` a um tensor de entrada. Devolve `(saida, nome_canonico)`."""
    fn, canonico = _resolve(net)
    return fn(entrada), canonico


def _entrada(board_size):
    return keras.Input(shape=(board_size, board_size, N_CHANNELS), name="board")


def _e_espacial(t):
    """True se o tronco devolveu um mapa `(H, W, C)` em vez de um vetor achatado."""
    return len(t.shape) == 4


def build_actor_critic(board_size=10, net="resnet_small", largura_densa=None,
                       n_actions=N_ACTIONS, nome=None):
    """Modelo de duas saídas `[logits, valor]` — o que PPO, A2C e ACER consomem.

    Em troncos que preservam a estrutura espacial (as ResNets), as cabeças são
    convoluções 1×1 seguidas de achatamento, como no AlphaZero: mais barato e mais
    informativo que jogar um `Dense` gigante em cima de um mapa achatado. Em troncos
    clássicos, que já achatam, usa-se a cabeça densa mesmo.

    O `Dense` final da política nasce com `kernel_initializer` de ganho pequeno: no início
    do treino a política precisa ser quase uniforme, senão o PPO gasta as primeiras
    iterações desfazendo uma preferência aleatória.
    """
    inp = _entrada(board_size)
    x, canonico = build_backbone(inp, net)
    largura = LARGURA_DENSA_PADRAO if largura_densa is None else int(largura_densa)

    if _e_espacial(x):
        p = layers.Conv2D(4, 1, use_bias=False, name="pi_c")(x)
        p = layers.GroupNormalization(groups=2, name="pi_n")(p)
        p = layers.Activation("relu", name="pi_a")(p)
        p = layers.Flatten(name="pi_f")(p)

        v = layers.Conv2D(2, 1, use_bias=False, name="v_c")(x)
        v = layers.GroupNormalization(groups=2, name="v_n")(v)
        v = layers.Activation("relu", name="v_a")(v)
        v = layers.Flatten(name="v_f")(v)
        v = layers.Dense(largura, activation="relu", name="v_d")(v)
    else:
        p = layers.Dense(largura, activation="relu", name="pi_d")(x)
        v = layers.Dense(largura, activation="relu", name="v_d")(x)

    logits = layers.Dense(
        n_actions, name="logits",
        kernel_initializer=keras.initializers.Orthogonal(gain=0.01),
        bias_initializer="zeros",
    )(p)
    valor = layers.Dense(
        1, name="value",
        kernel_initializer=keras.initializers.Orthogonal(gain=1.0),
        bias_initializer="zeros",
    )(v)

    return keras.Model(inp, [logits, valor], name=nome or f"ac_{canonico}")


def build_q_network(board_size=10, net="cnn_rainbow", largura_densa=None,
                    n_actions=N_ACTIONS, dueling=False, noisy=False, n_atoms=0,
                    nome=None):
    """A família DQN inteira num construtor só.

    `dueling`, `noisy` e `n_atoms` são os eixos que separam o DQN base do Rainbow — e são
    ortogonais de propósito, para que cada um possa ser medido isolado. Essa é a resposta
    aos seis notebooks quase idênticos do repositório antigo: uma função, seis chamadas.

    Saída
    -----
    `(lote, n_ações)` no modo normal; `(lote, n_ações, n_atoms)` de **logits** quando
    `n_atoms > 0` (C51).
    """
    inp = _entrada(board_size)
    x, canonico = build_backbone(inp, net)
    largura = LARGURA_DENSA_PADRAO if largura_densa is None else int(largura_densa)
    densa = "noisy" if noisy else "dense"

    if _e_espacial(x):
        x = layers.Conv2D(8, 1, use_bias=False, name="q_c")(x)
        x = layers.GroupNormalization(groups=2, name="q_n")(x)
        x = layers.Activation("relu", name="q_a")(x)
        x = layers.Flatten(name="q_f")(x)

    if n_atoms:
        saida = distributional_head(x, n_actions, n_atoms=n_atoms, largura=largura,
                                    densa=densa, dueling=dueling)
    elif dueling:
        saida = dueling_head(x, n_actions, largura=largura, densa=densa)
    else:
        h = _densa(densa, largura, "relu", "q_d")(x)
        saida = _densa(densa, n_actions, None, "q")(h)

    partes = [p for p, on in (("dueling", dueling), ("noisy", noisy),
                              (f"c51x{n_atoms}", bool(n_atoms))) if on]
    sufixo = ("_" + "_".join(partes)) if partes else ""
    return keras.Model(inp, saida, name=nome or f"q_{canonico}{sufixo}")


def build_policy_q(board_size=10, net="resnet_small", largura_densa=None,
                   n_actions=N_ACTIONS, nome=None):
    """Modelo de duas saídas `[logits, Q(s,·)]` — o que o ACER consome.

    Diferente do actor-critic comum: aqui o crítico devolve **um valor por ação**, não um
    escalar. É disso que o Retrace precisa, e `V(s) = Σ_a π(a|s) Q(s,a)` sai de graça —
    sem uma terceira cabeça e sem inconsistência entre V e Q, que é uma fonte clássica de
    bug silencioso em ACER.
    """
    inp = _entrada(board_size)
    x, canonico = build_backbone(inp, net)
    largura = LARGURA_DENSA_PADRAO if largura_densa is None else int(largura_densa)

    if _e_espacial(x):
        p = layers.Conv2D(4, 1, use_bias=False, name="pi_c")(x)
        p = layers.GroupNormalization(groups=2, name="pi_n")(p)
        p = layers.Activation("relu", name="pi_a")(p)
        p = layers.Flatten(name="pi_f")(p)

        q = layers.Conv2D(8, 1, use_bias=False, name="q_c")(x)
        q = layers.GroupNormalization(groups=2, name="q_n")(q)
        q = layers.Activation("relu", name="q_a")(q)
        q = layers.Flatten(name="q_f")(q)
        q = layers.Dense(largura, activation="relu", name="q_d")(q)
    else:
        p = layers.Dense(largura, activation="relu", name="pi_d")(x)
        q = layers.Dense(largura, activation="relu", name="q_d")(x)

    logits = layers.Dense(
        n_actions, name="logits",
        kernel_initializer=keras.initializers.Orthogonal(gain=0.01),
        bias_initializer="zeros",
    )(p)
    q_saida = layers.Dense(n_actions, name="q", bias_initializer="zeros")(q)
    return keras.Model(inp, [logits, q_saida], name=nome or f"acer_{canonico}")


def resumo(board_size=10, largura_densa=None):
    """Tabela comparativa dos troncos: parâmetros e formato de saída.

    Usada no notebook de ablação e no README. Revela, de graça, quais troncos colapsam o
    tabuleiro — a coluna `saída do tronco` mostra `1×1` para os que usam pooling.
    """
    linhas = []
    for nome in sorted(TRONCOS):
        inp = _entrada(board_size)
        saida, _ = build_backbone(inp, nome)
        forma = tuple(saida.shape[1:])
        modelo = build_actor_critic(board_size, nome, largura_densa)
        tronco = keras.Model(inp, saida)
        linhas.append({
            "tronco": nome,
            "saida_tronco": "×".join(str(d) for d in forma),
            "espacial": _e_espacial(saida),
            "params_tronco": tronco.count_params(),
            "params_actor_critic": modelo.count_params(),
        })
    return linhas


# --- snakeai/agents/base.py ---
"""Andaime comum a todos os agentes.

O que fica aqui é o que **precisa** ser idêntico entre algoritmos para que a comparação
valha: a cadência da avaliação, o formato do registro, o critério de "melhor checkpoint",
e os agendamentos lineares. O que varia — como o agente aprende — fica em cada módulo.

Foi essa separação que faltou no repositório antigo: cada notebook tinha o próprio laço de
treino, a própria noção de época e o próprio jeito de avaliar, e por isso as curvas nunca
puderam ser sobrepostas.
"""


import json
import os
from collections import deque
from dataclasses import asdict, dataclass, field

import numpy as np


__all__ = ["BaseConfig", "AgentBase"]


@dataclass
class BaseConfig:
    """Os campos que todo agente do benchmark tem. Cada algoritmo estende com os seus."""

    board_size: int = CONTRATO["board_size"]
    net: str = "resnet_small"
    seed: int = 0

    #: Orçamento oficial. O contrato exige o **mesmo** valor para todos os algoritmos.
    total_steps: int = 5_000_000

    #: Avaliação periódica durante o treino, no protocolo oficial.
    eval_every_steps: int = 250_000
    eval_episodes: int = CONTRATO["eval_episodes"]
    eval_envs: int = 250

    ckpt_dir: str = "checkpoints"
    runs_dir: str = "runs"
    log_every_steps: int = 50_000

    #: Artefatos gerados no fim do treino. O GIF custa segundos e responde a pergunta que
    #: nenhuma curva responde: *como* o agente joga.
    salvar_grafico: bool = True
    salvar_gif: bool = True
    gif_seeds: tuple = (7, 21, 42)

    def __post_init__(self):
        if self.board_size != CONTRATO["board_size"]:
            raise ValueError(
                f"board_size={self.board_size} viola o contrato "
                f"({CONTRATO['board_size']}). Mude o contrato conscientemente, "
                "não a execução."
            )


class AgentBase:
    """Laço de treino comum: agendamentos, avaliação, checkpoint e registro.

    A subclasse implementa `iterate()` — um passo de aprendizado, que devolve estatísticas
    do rollout — e o resto vem de graça, igual para todo mundo.
    """

    algo = "base"

    def __init__(self, cfg, variant="default"):
        self.cfg = cfg
        self.variant = variant
        self.model = None
        self.global_step = 0
        self.episodes = 0
        self.iteration = 0
        self.history = []
        self.evals = []
        self.baseline = None
        self.melhor = -np.inf
        self._proximo_eval = 0
        self._proximo_log = 0
        #: Janela de episódios recentes para a média móvel do treino. Sem ela, o log
        #: imprime a média dos episódios que por acaso terminaram **naquela** iteração —
        #: uma amostra de tamanho 0 a 3. É o que produzia a sequência
        #: `2,50 · 10,00 · — · — · 2,00 · 11,00`, que parece instabilidade do algoritmo e
        #: é só tamanho de amostra. O `—` é literalmente "nenhum episódio acabou agora".
        self._janela = deque(maxlen=200)
        os.makedirs(cfg.ckpt_dir, exist_ok=True)

    # ----------------------------------------------------------- agendamentos
    def media_movel(self):
        """Score médio dos episódios recentes, ponderado pela quantidade em cada iteração.

        `None` só quando nenhum episódio terminou na janela inteira — o que, com 200
        iterações, significa que o agente está mesmo sem terminar episódio.
        """
        n = sum(k for _, k in self._janela)
        return sum(soma for soma, _ in self._janela) / n if n else None

    def frac(self):
        """Fração do orçamento já gasta, em [0, 1]. Base de todo agendamento linear."""
        return min(1.0, self.global_step / max(1, self.cfg.total_steps))

    def linear(self, inicio, fim):
        return inicio + self.frac() * (fim - inicio)

    # -------------------------------------------------------------- avaliação
    def politica(self):
        """A função de política que `snakeai.eval` consome. Sobrescreva se precisar."""
        return keras_policy(self.model)

    def avaliar(self, episodes=None, safety=False):
        """Roda o protocolo oficial. **Nunca** com exploração — é o número honesto."""
        stats, _ = evaluate(
            self.politica(),
            board_size=self.cfg.board_size,
            episodes=episodes or self.cfg.eval_episodes,
            num_envs=self.cfg.eval_envs,
            greedy=CONTRATO["eval_greedy"],
            safety=safety,
            seed=CONTRATO["eval_seed"],
        )
        return stats

    def piso(self):
        if self.baseline is None:
            self.baseline = random_baseline(
                self.cfg.board_size, self.cfg.eval_episodes, self.cfg.eval_envs,
                seed=CONTRATO["eval_seed"],
            )
        return self.baseline

    # ------------------------------------------------------------- checkpoint
    def _caminho(self, tag, ext):
        return os.path.join(self.cfg.ckpt_dir, f"{self.algo}_{tag}.{ext}")

    def salvar(self, tag="last"):
        self.model.save(self._caminho(tag, "keras"))
        estado = {
            "global_step": self.global_step, "episodes": self.episodes,
            "iteration": self.iteration, "history": self.history,
            "evals": self.evals, "baseline": self.baseline, "melhor": self.melhor,
            "config": asdict(self.cfg), "variant": self.variant,
        }
        with open(self._caminho(tag, "json"), "w", encoding="utf-8") as f:
            json.dump(estado, f, ensure_ascii=False)

    def retomar(self, tag="last"):
        """Retoma do checkpoint. O Colab derruba a sessão — é questão de quando."""
        import keras

        m, s = self._caminho(tag, "keras"), self._caminho(tag, "json")
        if not (os.path.exists(m) and os.path.exists(s)):
            return False
        self.model = keras.models.load_model(m)
        self.on_model_reloaded()
        with open(s, encoding="utf-8") as f:
            estado = json.load(f)
        self.global_step = estado["global_step"]
        self.episodes = estado["episodes"]
        self.iteration = estado["iteration"]
        self.history = estado["history"]
        self.evals = estado.get("evals", [])
        self.baseline = estado.get("baseline")
        self.melhor = estado.get("melhor", -np.inf)
        self._proximo_eval = self.global_step + self.cfg.eval_every_steps
        self._proximo_log = self.global_step
        return True

    def on_model_reloaded(self):
        """Gancho: o otimizador antigo aponta para as variáveis do modelo antigo."""

    # ------------------------------------------------------------------ treino
    def iterate(self):
        raise NotImplementedError

    def train(self, verbose=True, ate_passos=None):
        """Roda até o orçamento, avaliando na cadência oficial. Devolve o `RunRecord`."""
        alvo = ate_passos or self.cfg.total_steps
        rec = Recorder(self.algo, variant=self.variant, seed=self.cfg.seed,
                       net=self.cfg.net,
                       params=self.model.count_params() if self.model else 0,
                       config=asdict(self.cfg), root=self.cfg.runs_dir)
        self.piso()

        while self.global_step < alvo:
            stats = self.iterate()
            self.iteration += 1

            m, k = stats.get("train_score_mean"), stats.get("n_episodes") or 0
            if m is not None and k:
                self._janela.append((m * k, k))

            if self.global_step >= self._proximo_log:
                self._proximo_log = self.global_step + self.cfg.log_every_steps
                # a curva registra a **média móvel**, não a iteração isolada: é o número
                # que responde "o treino está andando?" sem depender de quantos episódios
                # acabaram no exato momento do log
                ponto = {"episodes": self.episodes,
                         "train_score_mean": self.media_movel(),
                         "train_score_iter": stats.get("train_score_mean"),
                         **{k: v for k, v in stats.items() if k != "train_score_mean"}}
                self.history.append({"global_step": self.global_step, **ponto})
                rec.log(self.global_step, **ponto)
                if verbose:
                    self._imprimir(stats)

            if self.global_step >= self._proximo_eval:
                self._proximo_eval = self.global_step + self.cfg.eval_every_steps
                av = self.avaliar()
                av["global_step"] = self.global_step
                av["episodes"] = self.episodes
                self.evals.append(av)
                rec.log(self.global_step, eval_score_mean=av["score_mean"],
                        eval_score_p95=av["score_p95"], episodes=self.episodes)
                if verbose:
                    print(f"  [eval] passo {self.global_step:,} · "
                          f"score {av['score_mean']:.2f} "
                          f"(piso {self.baseline:.2f})")
                if av["score_mean"] > self.melhor:
                    self.melhor = av["score_mean"]
                    self.salvar("best")
                self.salvar("last")

        final = self.avaliar()
        rec.log(self.global_step, eval_score_mean=final["score_mean"],
                eval_score_p95=final["score_p95"], episodes=self.episodes)
        rec.finish(final)
        rec.record.meta["baseline"] = self.baseline
        self.salvar("last")

        # O registro é gravado SEMPRE. Estourar no fim de um treino de horas e perder a
        # curva seria o pior desfecho possível; o portão do contrato age na hora de
        # montar a arena, não na hora de escrever. As violações ficam no metadado e
        # `RunRecord.oficial` passa a ser False.
        problemas = validate(rec.record)
        if problemas:
            rec.record.meta["contract_violations"] = problemas
            if verbose:
                print("\n[contrato] esta execução NÃO entra na arena:")
                for p in problemas:
                    print(f"  - {p}")
        caminho = rec.save(skip_validation=True)
        if verbose:
            print(f"[registro] {caminho}")

        self.artefatos(rec, verbose=verbose)
        return rec

    # ---------------------------------------------------------------- artefatos
    def artefatos(self, rec, verbose=True):
        """Gráfico de diagnóstico e GIFs do agente jogando.

        Ficam ao lado do `history.json`, na pasta da execução — assim um checkpoint
        antigo nunca fica órfão da imagem que o explicava.
        """
        import os

        destino = os.path.dirname(rec.save(skip_validation=True))
        saida = {}

        if self.cfg.salvar_grafico:
            try:
                import matplotlib
                matplotlib.use("Agg")

                fig, _ = plot_run(rec.record)
                caminho = os.path.join(destino, "curva.png")
                fig.savefig(caminho, dpi=150, facecolor=fig.get_facecolor())
                matplotlib.pyplot.close(fig)
                saida["grafico"] = caminho
            except Exception as e:                      # nunca derrubar o treino por isso
                saida["grafico_erro"] = repr(e)

        if self.cfg.salvar_gif:

            politica = self.politica()
            for seed in self.cfg.gif_seeds:
                try:
                    caminho, score, motivo = render_episode(
                        politica, caminho=os.path.join(destino, f"episodio_s{seed}.gif"),
                        board_size=self.cfg.board_size, seed=seed,
                    )
                    saida[f"gif_s{seed}"] = {"caminho": caminho, "score": score,
                                             "fim": motivo}
                    if verbose:
                        print(f"[gif] seed {seed}: score {score}, terminou por {motivo}")
                except Exception as e:
                    saida[f"gif_s{seed}_erro"] = repr(e)

        rec.record.meta["artefatos"] = saida
        rec.save(skip_validation=True)
        return saida

    def _imprimir(self, stats):
        """Uma linha por log. Média móvel, não a iteração isolada — ver `self._janela`."""
        m = self.media_movel()
        m = f"{m:.2f}" if m is not None else "—"
        n = sum(k for _, k in self._janela)
        print(f"passo {self.global_step:>10,} · ep {self.episodes:>8,} · "
              f"treino {m:>6} (média de {n} episódios)")


# --- snakeai/memory/trajectory.py ---
"""Memória de **trajetórias** — o que o ACER precisa e o replay do DQN não dá.

O DQN sorteia transições soltas: para o alvo de TD, `(s, a, r, s')` basta. O ACER não —
o Retrace(λ) é uma recursão para trás no tempo, e a correção por importance sampling
compara a política atual com a que *gerou aquela sequência*. Sem a ordem temporal e sem a
política de comportamento gravada, nenhuma das duas coisas existe.

Foi exatamente aqui que o ACER legado quebrou. O erro era
``expected shape=(None, 256, 100), found shape=(None, 100)``: a dimensão de tempo tinha
sumido em algum ponto entre a coleta e o update. Aqui os segmentos são guardados com forma
`(T, N, ...)` explícita, e o teste `test_stored_segment_keeps_the_time_axis` trava isso.
"""


import numpy as np

__all__ = ["TrajectoryBuffer"]


class TrajectoryBuffer:
    """Guarda segmentos de rollout inteiros, com a política de comportamento.

    Cada entrada é um segmento `(T, N, ...)`: `T` passos de `N` ambientes em paralelo.
    Amostrar devolve um segmento inteiro, não uma transição — é a unidade que o Retrace
    consome.

    O campo `mu` é o que torna o algoritmo *off-policy* honesto: a probabilidade que a
    política tinha **no momento da coleta**. A razão `π/μ` sem esse registro seria `π/π`,
    ou seja, 1, e o ACER viraria um A2C caro.
    """

    def __init__(self, capacity, rng=None):
        self.capacity = int(capacity)
        self.dados = []
        self.pos = 0
        self.rng = rng if rng is not None else np.random.default_rng(0)

    def __len__(self):
        return len(self.dados)

    def add(self, obs, mask, act, mu, rew, done, obs_final, mask_final):
        """Guarda um segmento. Todos os arrays têm que começar com `(T, N, ...)`.

        `obs_final` / `mask_final` são o estado **logo depois** do último passo do
        segmento. Sem eles, o bootstrap do Retrace num segmento antigo teria de usar o
        estado atual do ambiente — que não tem relação nenhuma com aquela trajetória. É um
        erro que não levanta exceção: o algoritmo treina e aprende o valor errado.
        """
        T, N = act.shape
        for nome, arr, forma in (
            ("obs", obs, (T, N)), ("mask", mask, (T, N)), ("mu", mu, (T, N)),
            ("rew", rew, (T, N)), ("done", done, (T, N)),
        ):
            if arr.shape[:2] != forma:
                raise ValueError(
                    f"`{nome}` tem forma {arr.shape}; o eixo de tempo precisa vir "
                    f"primeiro: esperado {forma} + resto"
                )

        segmento = {
            "obs_final": np.asarray(obs_final, dtype=np.float32),
            "mask_final": np.asarray(mask_final, dtype=bool),
            "obs": np.asarray(obs, dtype=np.float32),
            "mask": np.asarray(mask, dtype=bool),
            "act": np.asarray(act, dtype=np.int32),
            "mu": np.asarray(mu, dtype=np.float32),
            "rew": np.asarray(rew, dtype=np.float32),
            "done": np.asarray(done, dtype=np.float32),
        }
        if len(self.dados) < self.capacity:
            self.dados.append(segmento)
        else:
            self.dados[self.pos] = segmento
        self.pos = (self.pos + 1) % self.capacity
        return segmento

    def sample(self):
        """Um segmento aleatório, com o eixo de tempo intacto."""
        if not self.dados:
            raise RuntimeError("memória de trajetórias vazia")
        return self.dados[int(self.rng.integers(0, len(self.dados)))]


# --- snakeai/agents/acer.py ---
"""ACER — *Actor-Critic with Experience Replay*.

O algoritmo que o repositório antigo tentou três vezes e nunca fez rodar. Reescrito do
zero em Keras 3, com as quatro peças que o definem:

1. **Retrace(λ)** para estimar o retorno a partir de dados velhos, com a recursão para trás
   no tempo e os pesos de importância truncados em 1.
2. **Gradiente de política com IS truncado + correção de viés.** Truncar a razão `π/μ`
   controla a variância mas introduz viés; o segundo termo devolve a parte cortada,
   somando sobre todas as ações. É o que torna o ACER não-enviesado *e* estável.
3. **Região de confiança contra a política média.** Uma cópia Polyak-média da política
   serve de âncora: se o passo proposto afastaria demais a política dela, ele é projetado
   de volta. Sem isso o ACER diverge com dados off-policy.
4. **Replay ratio.** Um update on-policy seguido de `k` updates sobre trajetórias
   guardadas — é daí que vem a eficiência amostral que justifica o algoritmo.

Os dois bugs legados, e por que não podem voltar
-------------------------------------------------
O ACER do `colab-rl` morria de duas formas distintas:

* ``TypeError: You are passing KerasTensor(...) to a TF API that does not allow
  registering custom dispatchers`` — a lógica do ACER estava sendo montada **dentro do
  grafo funcional do Keras**, onde os tensores são simbólicos. Aqui toda a matemática
  acontece em `tf.function` sobre tensores concretos; o modelo é só `entrada -> [logits,
  Q]`, e nada mais.
* ``ValueError: expected shape=(None, 256, 100), found shape=(None, 100)`` — a dimensão
  de tempo tinha se perdido. Aqui o rollout é `(T, N, ...)` explícito do começo ao fim, e
  o `TrajectoryBuffer` recusa qualquer coisa com outra forma.

Critério de desistência
-----------------------
ACER é o algoritmo mais difícil deste repositório e o que tem mais chance de não convergir
neste domínio. Se depois de um esforço delimitado ele não superar o piso aleatório de forma
consistente, a curva entra no benchmark assim mesmo, com a nota — um resultado negativo
medido vale mais que uma pasta chamada "Not Working".
"""


import os
from dataclasses import dataclass

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
import numpy as np
import tensorflow as tf


__all__ = ["ACERConfig", "ACER", "retrace"]


@dataclass
class ACERConfig(BaseConfig):
    num_envs: int = 64
    rollout: int = 32

    gamma: float = 0.995
    lr: float = 7e-4
    max_grad_norm: float = 10.0

    #: Truncamento do peso de importância no Retrace. O paper usa 1.
    c_retrace: float = 1.0
    #: Truncamento no gradiente de política. Acima disso entra a correção de viés.
    c_trunc: float = 10.0

    ent_coef: float = 0.01
    q_coef: float = 0.5

    #: Região de confiança: raio `delta` e taxa da média de Polyak da política âncora.
    trust_region: bool = True
    delta: float = 1.0
    polyak: float = 0.99

    #: Updates off-policy por update on-policy. É a razão de existir do ACER.
    replay_ratio: int = 4
    memory_size: int = 500
    warmup_segments: int = 8


def retrace(rew, done, q_a, v, rho_barra, ultimo_v, gamma):
    """Alvo Retrace(λ), recursão para trás no tempo.

    `Q^ret_t = r_t + γ (1−d_t) [ ρ̄_{t+1} (Q^ret_{t+1} − Q(s_{t+1}, a_{t+1})) + V(s_{t+1}) ]`

    Cortar o peso de importância em 1 (`ρ̄ = min(1, ρ)`) é o que torna o estimador seguro
    para dados arbitrariamente velhos: a correção nunca amplifica, só encolhe. É por isso
    que o ACER pode reusar trajetórias que um A2C teria de jogar fora.

    Formas: tudo `(T, N)`; `ultimo_v` é `(N,)`.
    """
    T, N = rew.shape
    ret = np.zeros((T, N), dtype=np.float32)
    prox_ret = ultimo_v.astype(np.float32)
    prox_q = ultimo_v.astype(np.float32)
    prox_v = ultimo_v.astype(np.float32)
    prox_rho = np.ones(N, dtype=np.float32)

    for t in reversed(range(T)):
        continua = 1.0 - done[t]
        ret[t] = rew[t] + gamma * continua * (
            prox_rho * (prox_ret - prox_q) + prox_v
        )
        prox_ret = ret[t]
        prox_q = q_a[t]
        prox_v = v[t]
        prox_rho = rho_barra[t]
    return ret


class ACER(AgentBase):
    algo = "acer"

    def __init__(self, cfg: ACERConfig = None, variant=None):
        cfg = cfg or ACERConfig()
        super().__init__(cfg, variant=variant or cfg.net)
        keras.utils.set_random_seed(cfg.seed)

        self.model = build_policy_q(cfg.board_size, cfg.net)
        self.media = build_policy_q(cfg.board_size, cfg.net)     # política âncora
        self.media.set_weights(self.model.get_weights())

        self.optimizer = keras.optimizers.Adam(cfg.lr, clipnorm=cfg.max_grad_norm)
        self.optimizer.build(self.model.trainable_variables)

        self.env = VecSnake(cfg.num_envs, cfg.board_size,
                            rng=np.random.default_rng(cfg.seed))
        self.obs, self.mask = self.env.reset()
        self.rng = np.random.default_rng(cfg.seed + 1)
        self.memoria = TrajectoryBuffer(cfg.memory_size,
                                        rng=np.random.default_rng(cfg.seed + 2))

    def on_model_reloaded(self):
        self.media = keras.models.clone_model(self.model)
        self.media.set_weights(self.model.get_weights())
        self.optimizer = keras.optimizers.Adam(self.cfg.lr,
                                               clipnorm=self.cfg.max_grad_norm)
        self.optimizer.build(self.model.trainable_variables)

    # ------------------------------------------------------------------ política
    def politica(self):
        """Greedy sobre os logits — a mesma régua dos outros agentes."""
        @tf.function(reduce_retracing=True)
        def frente(obs, mask):
            logits, _ = self.model(obs, training=False)
            return tf.where(mask, logits, tf.fill(tf.shape(logits), MASK_NEG))

        def fn(obs, mask):
            return frente(tf.convert_to_tensor(obs), tf.convert_to_tensor(mask)).numpy()
        return fn

    @tf.function(reduce_retracing=True)
    def _probs(self, obs, mask):
        logits, q = self.model(obs, training=False)
        logits = tf.where(mask, logits, tf.fill(tf.shape(logits), MASK_NEG))
        return tf.nn.softmax(logits), q

    # ------------------------------------------------------------------ rollout
    def collect(self):
        cfg = self.cfg
        T, N, b = cfg.rollout, cfg.num_envs, cfg.board_size

        obs_buf = np.empty((T, N, b, b, N_CHANNELS), dtype=np.float32)
        mask_buf = np.empty((T, N, N_ACTIONS), dtype=bool)
        act_buf = np.empty((T, N), dtype=np.int32)
        mu_buf = np.empty((T, N, N_ACTIONS), dtype=np.float32)
        rew_buf = np.empty((T, N), dtype=np.float32)
        done_buf = np.empty((T, N), dtype=np.float32)

        scores, vitorias = [], 0
        for t in range(T):
            obs_buf[t], mask_buf[t] = self.obs, self.mask
            pi, _ = self._probs(tf.convert_to_tensor(self.obs),
                                tf.convert_to_tensor(self.mask))
            pi = pi.numpy()
            mu_buf[t] = pi
            a = (pi.cumsum(1) > self.rng.random((N, 1))).argmax(1).astype(np.int32)
            act_buf[t] = a

            self.obs, self.mask, r, d, info = self.env.step(a)
            rew_buf[t], done_buf[t] = r, d.astype(np.float32)
            scores.extend(info["scores"].tolist())
            vitorias += info["wins"]

        self.global_step += T * N
        self.episodes += len(scores)
        # o estado logo após o último passo — é ele que faz o bootstrap do Retrace deste
        # segmento, hoje e daqui a mil iterações
        segmento = self.memoria.add(obs_buf, mask_buf, act_buf, mu_buf, rew_buf, done_buf,
                                    obs_final=self.obs.copy(), mask_final=self.mask.copy())
        stats = {
            "train_score_mean": float(np.mean(scores)) if scores else None,
            "n_episodes": len(scores),
            "wins": vitorias,
            "segmentos": len(self.memoria),
        }
        return segmento, stats

    # ------------------------------------------------------------------- update
    @tf.function(reduce_retracing=True)
    def _passo(self, obs, mask, act, mu, ret, ent_coef, q_coef, c_trunc, delta,
               trust_region):
        """Um passo de gradiente do ACER, em tensores concretos.

        Tudo aqui é `tf.function` sobre tensores reais — nunca `KerasTensor` dentro do
        grafo funcional, que era o `TypeError` que matava o ACER legado.
        """
        with tf.GradientTape() as tape_ext:
            logits, q = self.model(obs, training=True)
            logits = tf.where(mask, logits, tf.fill(tf.shape(logits), MASK_NEG))
            pi = tf.nn.softmax(logits)
            logpi = tf.nn.log_softmax(logits)

            v = tf.reduce_sum(pi * q, axis=-1)                     # V = Σ π Q
            q_a = tf.gather(q, act, batch_dims=1)
            pi_a = tf.gather(pi, act, batch_dims=1)
            mu_a = tf.maximum(tf.gather(mu, act, batch_dims=1), 1e-8)
            rho = pi_a / mu_a
            rho_todas = pi / tf.maximum(mu, 1e-8)

            vantagem_ret = tf.stop_gradient(ret - v)
            vantagem_q = tf.stop_gradient(q - tf.expand_dims(v, -1))

            with tf.GradientTape() as tape_logits:
                tape_logits.watch(logits)
                lp = tf.nn.log_softmax(logits)
                lp_a = tf.gather(lp, act, batch_dims=1)

                # termo 1: IS truncado na ação tomada
                termo1 = tf.minimum(c_trunc, tf.stop_gradient(rho)) * lp_a * vantagem_ret
                # termo 2: correção de viés — devolve a parte que o truncamento cortou,
                # somando sobre TODAS as ações, ponderada pela política atual
                corte = tf.nn.relu(1.0 - c_trunc / tf.maximum(
                    tf.stop_gradient(rho_todas), 1e-8))
                termo2 = tf.reduce_sum(
                    corte * tf.stop_gradient(pi) * lp * vantagem_q, axis=-1)
                objetivo = tf.reduce_mean(termo1 + termo2)

            g = tape_logits.gradient(objetivo, logits)

            if trust_region:
                logits_media, _ = self.media(obs, training=False)
                logits_media = tf.where(mask, logits_media,
                                        tf.fill(tf.shape(logits_media), MASK_NEG))
                pi_media = tf.nn.softmax(logits_media)
                # k = ∇_logits KL(π_média || π) = π − π_média
                k = pi - pi_media
                kg = tf.reduce_sum(k * g, axis=-1, keepdims=True)
                kk = tf.reduce_sum(k * k, axis=-1, keepdims=True) + 1e-8
                escala = tf.nn.relu((kg - delta) / kk)
                z = g - escala * k
            else:
                z = g

            # devolve o gradiente projetado para a rede: d(perda)/d(logits) = −z
            perda_pi = -tf.reduce_sum(tf.stop_gradient(z) * logits)
            perda_q = q_coef * tf.reduce_mean(tf.square(tf.stop_gradient(ret) - q_a))
            entropia = -tf.reduce_mean(tf.reduce_sum(pi * logpi, axis=-1))
            perda = perda_pi + perda_q - ent_coef * entropia

        grads = tape_ext.gradient(perda, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
        return perda_q, entropia, tf.reduce_mean(rho)

    def _alvo_retrace(self, seg):
        """Recalcula π e Q com a rede **atual** sobre a trajetória guardada."""
        cfg = self.cfg
        T, N = seg["act"].shape
        obs = seg["obs"].reshape(T * N, *seg["obs"].shape[2:])
        mask = seg["mask"].reshape(T * N, N_ACTIONS)

        pi, q = self._probs(tf.convert_to_tensor(obs), tf.convert_to_tensor(mask))
        pi = pi.numpy().reshape(T, N, N_ACTIONS)
        q = q.numpy().reshape(T, N, N_ACTIONS)

        v = (pi * q).sum(-1)
        idx_t, idx_n = np.indices((T, N))
        q_a = q[idx_t, idx_n, seg["act"]]
        pi_a = pi[idx_t, idx_n, seg["act"]]
        mu_a = np.maximum(seg["mu"][idx_t, idx_n, seg["act"]], 1e-8)
        rho_barra = np.minimum(cfg.c_retrace, pi_a / mu_a).astype(np.float32)

        # bootstrap com o estado final DO SEGMENTO, nunca com o estado atual do ambiente:
        # num update off-policy os dois não têm relação nenhuma
        pi_f, q_f = self._probs(tf.convert_to_tensor(seg["obs_final"]),
                                tf.convert_to_tensor(seg["mask_final"]))
        ultimo_v = (pi_f.numpy() * q_f.numpy()).sum(-1).astype(np.float32)

        ret = retrace(seg["rew"], seg["done"], q_a, v, rho_barra, ultimo_v, cfg.gamma)
        return ret

    def _aprender(self, seg):
        cfg = self.cfg
        T, N = seg["act"].shape
        ret = self._alvo_retrace(seg)

        perda_q, ent, rho = self._passo(
            tf.convert_to_tensor(seg["obs"].reshape(T * N, *seg["obs"].shape[2:])),
            tf.convert_to_tensor(seg["mask"].reshape(T * N, N_ACTIONS)),
            tf.convert_to_tensor(seg["act"].reshape(T * N)),
            tf.convert_to_tensor(seg["mu"].reshape(T * N, N_ACTIONS)),
            tf.convert_to_tensor(ret.reshape(T * N)),
            cfg.ent_coef, cfg.q_coef, cfg.c_trunc, cfg.delta, cfg.trust_region,
        )

        # média de Polyak da política âncora
        if cfg.trust_region:
            p = cfg.polyak
            self.media.set_weights([
                p * a + (1.0 - p) * b
                for a, b in zip(self.media.get_weights(), self.model.get_weights())
            ])
        return float(perda_q), float(ent), float(rho)

    # -------------------------------------------------------------------- passo
    def iterate(self):
        cfg = self.cfg
        seg, stats = self.collect()

        perdas = [self._aprender(seg)]                      # on-policy
        if len(self.memoria) >= cfg.warmup_segments:        # off-policy
            for _ in range(cfg.replay_ratio):
                perdas.append(self._aprender(self.memoria.sample()))

        q, e, r = (float(np.mean(x)) for x in zip(*perdas))
        stats.update({"loss_q": q, "entropia": e, "rho_medio": r,
                      "updates": len(perdas)})
        return stats


# ==== FIM DO CÓDIGO GERADO ====

## Configuração

Os padrões abaixo são os do **contrato**: tabuleiro 10×10, 5 M passos de orçamento,
avaliação de 1.000 episódios com semente 123. Mexer neles é legítimo para experimentar,
mas o resultado só entra na arena se o contrato for respeitado — o `Recorder` recusa
qualquer outra coisa e diz o motivo.


In [ ]:
# @title Parâmetros
SEMENTE = 0        # @param {type:"integer"}
PASSOS = 5000000   # @param {type:"integer"}
REDE = "resnet_small"  # @param ["resnet_tiny", "resnet_small", "resnet_base", "cnn_rainbow", "cnn_alphazero", "cnn_vgg", "cnn_vgg_dropout", "cnn_vgg_sem_pool"]
USAR_DRIVE = True  # @param {type:"boolean"}

# Ligado por padrão: a sessão do Colab cai, e sem o Drive ela leva junto os
# checkpoints — o treino não retoma de onde parou, recomeça do zero.
PASTA = "/content/snake-arena"
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA = "/content/drive/MyDrive/snake-arena"

cfg = ACERConfig(
    seed=SEMENTE,
    net=REDE,
    total_steps=PASSOS,
    ckpt_dir=os.path.join(PASTA, "checkpoints"),
    runs_dir=os.path.join(PASTA, "runs"),
)
print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))

## Treino

**Retomável.** A sessão do Colab vai cair — é questão de quando, não de se. Rode a célula
de novo e ela continua do último checkpoint. Com `USAR_DRIVE = True` os checkpoints
sobrevivem à queda da máquina.


In [ ]:
# @title Treinar
agente = ACER(cfg)
if agente.retomar("last"):
    print("retomando do checkpoint")
print("parâmetros:", f"{agente.model.count_params():,}")

registro = agente.train(verbose=True)

## Veredito

Três regimes na mesma execução: o piso aleatório, a política pura e a política com o
filtro de segurança. Se a coluna do meio não estiver bem acima do piso, não aprendeu — e
aí o problema é hiperparâmetro ou tempo de treino, não código.


In [ ]:
# @title Veredito
resultado = verdict(agente.politica(), episodes=1000)
print(format_verdict(resultado))

fig, _ = plot_run(registro.record)
plt.show()

## O agente jogando

Um GIF vale mais que a curva para entender *como* o agente perde. Morrer preso no próprio
corpo e morrer de fome dão a mesma linha no gráfico e são problemas completamente
diferentes.


In [ ]:
# @title GIF
from IPython.display import Image, display

for semente in (7, 21, 42):
    caminho, score, motivo = render_episode(
        agente.politica(), caminho=f"episodio_s{semente}.gif", seed=semente)
    print(f"semente {semente}: score {score}, terminou por {motivo}")
    display(Image(filename=caminho))

## Exportar

`.keras` para retomar treino, TFLite fp16/int8 para embarcar no jogo. A paridade de **ação**
contra o `.keras` é conferida — diferença numérica de quantização é aceitável, ação
diferente não é.


In [ ]:
# @title Exportar
relatorio = export_model(agente.model, out_dir=os.path.join(PASTA, "export"))
print(json.dumps(relatorio, indent=2, ensure_ascii=False))

## Onde ficou o resultado

O `history.json` da execução vai para `runs/<algo>/<variante>/seed<N>/`, junto com a curva e
os GIFs. Essa pasta é o que entra na arena: coloque em `runs/` do repositório e rode
`python -m snakeai.arena --all`.


In [ ]:
# @title Conferir o contrato
CAMINHO_REGISTRO = registro.save(skip_validation=True)
print("registro:", CAMINHO_REGISTRO)

problemas = validate(registro.record)
print("entra na arena?" , "sim" if not problemas else "NÃO:")
for p in problemas:
    print("  -", p)

## Baixar o resultado

Um `.zip` só, com a pasta inteira da execução — registro, curva, GIFs e o modelo exportado.

**Um arquivo, e não vários downloads**, por dois motivos: o navegador bloqueia downloads
múltiplos disparados em sequência, e a pasta da execução só faz sentido inteira — o
`history.json` sem a curva e sem os GIFs perde metade do que ela responde.

Se a aba do Colab não estiver aberta na hora em que isso rodar, o download automático não
acontece (o navegador precisa estar lá para receber). Nesse caso o `.zip` fica salvo e a
célula imprime o caminho — dá para baixar pelo painel de arquivos à esquerda, ou pelo Drive
se `USAR_DRIVE = True`.


In [ ]:
# @title Baixar tudo num .zip
import shutil

PASTA_EXECUCAO = os.path.dirname(CAMINHO_REGISTRO)

# o export mora fora da pasta da execução; copiamos para dentro antes de zipar,
# senão o .zip sai sem o modelo — que é justamente o que se leva para o jogo
_export = os.path.join(PASTA, "export")
if os.path.isdir(_export):
    shutil.copytree(_export, os.path.join(PASTA_EXECUCAO, "export"), dirs_exist_ok=True)

_nome = "_".join([registro.record.algo, registro.record.variant,
                  f"seed{registro.record.seed}"])
ZIP = shutil.make_archive(os.path.join(PASTA, _nome), "zip", PASTA_EXECUCAO)
print(f"{ZIP}  ({os.path.getsize(ZIP) / 1e6:.1f} MB)")
for _raiz, _, _arqs in os.walk(PASTA_EXECUCAO):
    for _a in sorted(_arqs):
        print("   ", os.path.relpath(os.path.join(_raiz, _a), PASTA_EXECUCAO))

try:
    from google.colab import files
    files.download(ZIP)
except Exception as e:
    print()
    print(f"download automático não rolou ({type(e).__name__}: {e})")
    print(f"o .zip está em {ZIP} — baixe pelo painel de arquivos")